This is the main workbook, i will be using. The workbook should only have imported functions.
    

In [ ]:
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns


#import torch
#from torch.utils.data import Dataset, DataLoader
#from torchvision import transforms

#from PIL import Image
#import random
from pathlib import Path
import os


%load_ext autoreload
%autoreload 2

# 0 Become One with the data

## 0.1 Understanding the data

## Dataset Refinement Plan (Based on Section 03 Performance)

Because the results in Section 03 look poor, I am going to perform the following data cleaning and class consolidation steps:

### 1. Delete Class 1
* **Reason:** Data labeling mismatch. The training folder is labeled as `vug`, but the validation and test folders are labeled as `arenaceous`. Additionally, the visual characteristics of `arenaceous` look significantly different from `vug`.

---

### 2. Merge Class 13 and Class 2
* **Reason:** There is a massive, persistent, unidirectional confusion where Class 2 is completely absorbed by Class 13. Because this happens heavily in the training data, the model cannot find clean decision boundaries to distinguish them.
* **Confusion Metrics:**
  * **Train:** True Class 2 is predicted as Class 13 **83 times**.
  * **Validation:** True Class 2 is predicted as Class 13 **155 times**.
  * **Test:** True Class 2 is predicted as Class 13 **81 times**.

---

### 3. Merge Class 21 and Class 22
* **Reason:** High mutual confusion and a "triangle of confusion" involving Class 1. At a minimum, merging these two will consolidate the bleeding boundaries.
* **Confusion Metrics:**
  * **Train:** 
    * True Class 21 bleeds heavily into Class 1 (**97 times**) and Class 22 (**82 times**).
    * True Class 22 bleeds into Class 1 (**93 times**) and Class 21 (**80 times**).
  * **Validation / Test:** This exact same triangle of confusion persists proportionately across both sets.

---

### 4. Merge Class 18 and Class 19
* **Reason:** These two classes show significant mutual confusion across the board, making them excellent candidates for consolidation.
* **Confusion Metrics:**
  * **Train:** True Class 18 predicted as Class 19 (**41 times**) | True Class 19 predicted as Class 18 (**24 times**).
  * **Validation:** True Class 18 predicted as Class 19 (**41 times**) | True Class 19 predicted as Class 18 (**71 times**).
  * **Test:** True Class 18 predicted as Class 19 (**35 times**) | True Class 19 predicted as Class 18 (**45 times**).


In [ ]:
data_path = Path("data/carbonate_1223 _merge")
image_path = data_path / "PPL-1223"

In [ ]:
# delete class 1
from modular.data_setup import delete_class
delete_class(root_dir=image_path,class_name="class1")

In [ ]:
# merge class 2 and 13
from modular.data_setup import merge_classes
merge_classes(root_dir=image_path,class_a="class2", class_b="class13",new_class_name="class2_class13")

In [ ]:
# merge class 18 and 19
merge_classes(root_dir=image_path,class_a="class18",class_b="class19",new_class_name="class18_class19")

In [ ]:
# merge class 21 and 22
merge_classes(root_dir=image_path,class_a="class21",class_b="class22",new_class_name="class21_class22")

In [ ]:
from modular.data_setup import walk_through_dir
counts_df=walk_through_dir(image_path)
counts_df

### Plot the distrubition

In [ ]:
# Execute the function
from modular.visualization import plot_awesome_barchart
plot_awesome_barchart(counts_df)


### Strategies for Handling Class Imbalance

Given the heavy long-tail distribution in this dataset, standard training approaches will likely overfit to the majority classes. To build a robust model, we will combine several approaches across the training pipeline:

#### 1. Data Pipeline & Sampling
*   **Targeted Augmentation:** Apply heavier or specific image augmentations (e.g., rotations, color jitter, cropping) exclusively to the minority classes to artificially increase their variance and representation.
*   **`WeightedRandomSampler`:** Utilize a weighted sampler during the data loading phase to oversample minority classes. This ensures each training batch has a more balanced class distribution.

#### 2. Modeling & Loss Strategy
*   **Transfer Learning:** Fine-tune a pre-trained model rather than training from scratch. Pre-trained weights provide robust feature extractors that help the network generalize even with limited data in rare classes.
*   **Class-Weighted Cross-Entropy:** Apply weights to the loss function that are inversely proportional to class frequencies. This forces the optimizer to treat mistakes on rare classes as more severe than mistakes on majority classes.

#### 3. Evaluation Metrics
*   **Robust Metrics:** Standard accuracy is easily inflated by majority classes. Model selection should be based on **macro F1-score**, **balanced accuracy**, and **per-class recall**.
*   **Confusion Matrix:** Carefully examine the confusion matrix after validation epochs to identify if specific minority classes are being consistently misclassified as majority classes.

#### 4. Dataset Curation
*   **Class Merging:** Re-evaluate the rarest classes (e.g., those with fewer than 100 samples). If they lack distinct geological or feature-level support, consider merging them into visually or logically similar parent classes to improve model stability.

### Summary of Strategies for Imbalanced Datasets

| Approach | What it does | Good for your case? |
| :--- | :--- | :--- |
| **Collect more minority-class images** | Adds real information | Best solution |
| **Merge rare, geologically similar classes** | Reduces impossible classes | Very important |
| **Data augmentation** | Creates variations of minority images | Yes |
| **Weighted loss** | Penalizes minority-class mistakes more | Yes |
| **WeightedRandomSampler** | Shows minority classes more often | Yes |
| **Undersampling majority classes** | Reduces dominance of big classes | Sometimes |
| **Focal loss** | Focuses learning on difficult examples | Worth testing |
| **Transfer learning** | Requires less data than training from scratch | Definitely |
| **Synthetic generation** | Creates artificial minority images | Possible, but risky |
| **Hierarchical classification** | Predict broad group → subclass | Potentially excellent |
| **Better evaluation metrics** | Prevents misleading accuracy | Essential |

## 0.2 Making the dataset

In [ ]:
# Run this inside a notebook cell to delete all hidden checkpoints in your data folder
!find data/ -type d -name ".ipynb_checkpoints" -exec rm -rf {} +

### 0.2.1 Make the full dataset

In [ ]:
from modular.data_setup import RockClassificationDataset
train_dir = "data/carbonate_1223 _merge/PPL-1223/train"
val_dir   = "data/carbonate_1223 _merge/PPL-1223/val"
test_dir  = "data/carbonate_1223 _merge/PPL-1223/test"


train_dataset = RockClassificationDataset(root_dir=train_dir,image_transform=None)

val_dataset = RockClassificationDataset(root_dir=val_dir,image_transform=None)

test_dataset = RockClassificationDataset(root_dir=test_dir,image_transform=None)


### 0.2.2 Make smaller datasets

In [ ]:
# Import data_setup.py
from modular.data_setup import make_stratified_subset
labels = [label for _, label in train_dataset.samples]

train_subset_5_dataset = make_stratified_subset(train_dataset,labels,fraction=0.05,random_seed=42)
train_subset_20_dataset = make_stratified_subset(train_dataset,labels,fraction=0.20,random_seed=42)
train_subset_5_dataset

## 0.3 Visualize the data

In [ ]:
from modular.visualization import plot_random_image
plot_random_image("data/carbonate_1223 _merge/PPL-1223/train", seed=None)

### Dataset Audit: DeepCarbonate Quality & Consistency Review

An inspection of the **DeepCarbonate** dataset identified critical inconsistencies between the folder labels, image filenames, and geological classifications across all three optical modes (**PPL**, **XPL**, and **reflected light**).

---

#### ⚠️ Labeling & Filename Discrepancies

* **Class 1 Naming Mismatch:** The training images in `class1` are named `Vug.*`, whereas the validation and test images are named `Arenaceous.*`. However, the supplied class map identifies `class1` as **Microbial mudstone**.
* **Overlapping Filenames:** `class22` (identified as **Pore**) also contains images named `Vug.*`.
* **Index Non-Uniqueness:** In the **PPL training split**, **1,275 filenames** occur simultaneously in both `class1` and `class22`. 
    > 🔍 *Note: File-content and perceptual comparisons show that these are completely different images rather than exact or near duplicates. The filename index is therefore not globally unique.*

#### 🧠 Conceptual Inconsistencies

There is a fundamental mismatch in describing all 22 targets strictly as lithological classes:
* **Feature vs. Rock Type:** A **pore** is not a rock type; it is a petrographic or reservoir feature. 
* **Other Affected Categories:** Classes such as **cemented fracture** and **stylolite** represent diagenetic features rather than complete rock lithologies.

> **Revised Definition:** DeepCarbonate is more accurately described as a **petrographic image-category dataset** that combines lithologies, depositional textures, and reservoir features.

---

#### 🛠️ Actionable Pipeline Recommendations

1. **Folder-Based Pipelines:** Until the `class1` naming discrepancy is verified, class assignments **must be derived strictly from folder placement** rather than individual filenames (e.g., using PyTorch's `datasets.ImageFolder`). Do not use filenames to determine an image's class.
2. **Manual Visual Audits:** Representative images from `class1` should be manually reviewed to verify the actual structural characteristics.
3. **Model Architecture Adjustments:**
    * *To Benchmark:* Models reproducing the original benchmark can retain all 22 categories as-is.
    * *For Strict Rock Classifiers:* A pure rock-lithology classifier should either **exclude feature-based categories** entirely or treat them as separate **multi-label outputs**.


# 1. Transformations

## 1.0 Make the transformation functions

In [ ]:
# input u can put into for mean and std if u are important a model like what is below.
mean_imagenet=[0.485, 0.456, 0.406]
std_imagenet  = [0.229, 0.224, 0.225]

### Testing the transforms function

In [ ]:
from modular.data_setup import create_transformation
# Below is to create a transformation that will remove bottom part of pixel, crop,resize and do other transformation including taking mean and std of my dataset.

train_transform, val_test_transform, mean, std,dataset_len = create_transformation(
    train_dataset=train_dataset,normalization = "mydataset", pixels=90
)
print(f"train transform is \n{train_transform}")
print(f"val and test transform is \n{val_test_transform}")
print(f"mean is \n{mean}")
print(f"std \n{std}")
print(f"length of data is  \n{dataset_len}")

In [ ]:
train_transform_5, val_test_transform_5, mean_5, std_5,dataset_len_5 = create_transformation(train_dataset=train_subset_5_dataset,normalization = "mydataset", pixels=90)
print(f"train transform is \n{train_transform_5}")
print(f"val and test transform is \n{val_test_transform_5}")
print(f"mean is \n{mean_5}")
print(f"std is \n{std_5}")
print(f"length of data is  \n{dataset_len_5}")


In [ ]:
train_transform_20, val_test_transform_20, mean_20, std_20,dataset_len_20 = create_transformation(
    train_dataset=train_subset_20_dataset,normalization = "mydataset", pixels=90
)
print(f"train transform is \n{train_transform_20}")
print(f"val and test transform is \n{val_test_transform_20}")
print(f"mean is \n{mean_20}")
print(f"std \n{std_20}")
print(f"length of data is  \n{dataset_len_20}")


#### other things to try

| Transformation | Use? | Reason |
|---|---|---|
| **Resize** | ✅ | Required for batching/pretrained network |
| **RandomResizedCrop** | ✅ | Helps model not depend on exact framing |
| **Horizontal flip** | ✅ | Rock type usually shouldn't change if mirrored |
| **Vertical flip** | ⚠️ | Depends on whether geological orientation matters |
| **Rotation ±15°** | ✅ | Camera orientation shouldn't control classification |
| **Brightness** | ✅ | Handles illumination variation |
| **Contrast** | ✅ | Handles shadows/exposure |
| **Saturation** | ✅ mild | Handles camera/weather variation |
| **Hue** | ⚠️ very mild | Rock color can be diagnostic |
| **Gaussian blur** | ⚠️ | Can destroy grain/texture information |
| **Random erasing** | ⚠️ | Could hide diagnostic geological features |
| **Perspective distortion** | ❌ initially | Often physically unrealistic |
| **Huge rotation/distortion** | ❌ | May create unrealistic samples |

#### Experiment 1 — Minimal
* RandomResizedCrop
* HorizontalFlip
* Normalize
  
#### Experiment 1 — Minimal
* RandomResizedCrop
* verticalFlip
* Normalize

#### Experiment 2 — Recommended
* RandomResizedCrop
* HorizontalFlip
* Rotation ±15°
* Mild ColorJitter
* Normalize

#### Experiment 3 — Stronger
* RandomResizedCrop
* HorizontalFlip
* VerticalFlip
* Rotation ±20°
* ColorJitter
* RandomAffine
* Normalize

## 1.1 Veiw results of transformation

In [ ]:
image_path_list_5 = [
    train_subset_5_dataset.dataset.samples[i][0]
    for i in train_subset_5_dataset.indices
]

In [ ]:
from modular.visualization import plot_transformed_images

plot_transformed_images(
    image_paths=image_path_list_5,
    transform=train_transform_5,
    mean=mean_5,
    std=std_5,
    n=5,
    seed=None
)

# 2. Loading the dataset

## 2.1 Give each subset its OWN transformed dataset

In [ ]:
from modular.data_setup import subset_with_transform
train_subset_5_dataset = subset_with_transform(
    train_subset_5_dataset,
    train_transform_5
)
train_subset_20_dataset = subset_with_transform(
    train_subset_20_dataset,
    train_transform_20
)


## 5 % of the data

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_dataloader_5 = DataLoader(
    train_subset_5_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)
val_dataset_5 = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=val_test_transform_5
)
val_dataloader_5 = DataLoader(
    val_dataset_5,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [ ]:
print(train_subset_5_dataset.dataset.image_transform)
print(val_dataset_5.image_transform)

## 20 % of the data

In [ ]:

from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_dataloader_20 = DataLoader(
    train_subset_20_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_dataset_20 = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=val_test_transform_20
)

val_dataloader_20 = DataLoader(
    val_dataset_20,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [ ]:
print(train_subset_20_dataset.dataset.image_transform)
print(val_dataset_20.image_transform)

## All of the data

In [ ]:

from torch.utils.data import DataLoader
BATCH_SIZE=32
train_dataset.image_transform = train_transform
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_dataset = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=val_test_transform
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)
# test data
test_dataset = RockClassificationDataset(
    root_dir=test_dir,
    image_transform=val_test_transform
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [ ]:
print(train_dataset.image_transform)
print(val_dataset.image_transform)

# 3. Pre Models Needs

## 3.0 Check GPU

In [ ]:
import torch

In [ ]:
!nvidia-smi

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch CUDA version:", torch.version.cuda)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

## 3.1 Impot torch metrics

In [ ]:
from torchmetrics import MetricCollection 
from torchmetrics.classification import (MulticlassAccuracy,MulticlassF1Score,MulticlassPrecision,MulticlassRecall)

In [ ]:
class_names = train_dataset.classes
metrics = MetricCollection({
    "accuracy": MulticlassAccuracy(num_classes=len(class_names),average="micro"),
    "f1": MulticlassF1Score(num_classes=len(class_names),average="macro"),
    "precision": MulticlassPrecision(num_classes=len(class_names),average="macro"),
    "recall": MulticlassRecall(num_classes=len(class_names),average="macro")
})

In [ ]:
train_metrics = metrics.clone(prefix="train_").to(device)
test_metrics = metrics.clone(prefix="test_").to(device)

#### micro vs Marco

When evaluating a multiclass model on an imbalanced dataset, the choice between micro and macro averaging dictates whether you are giving equal weight to every image or every class.

* **Micro-Averaging (Image-focused):** Pools all predictions together before calculating the metric. Because it weighs every image equally, massive classes will dominate the final score. It answers: *"What percentage of total images were correct?"*
* **Macro-Averaging (Class-focused):** Calculates the metric for each class individually, then averages those class scores. It treats all classes equally, regardless of how many images they have. It answers: *"How well does the model perform across all classes on average?"*

**The Key Takeaway:** If you have an imbalanced dataset, a high Micro Accuracy can hide terrible performance on minority classes. Using Macro metrics (like Macro F1) exposes those weaknesses and provides a truer picture of the model's overall capability. 

If you run these metrics on your dashboard and see the following results:

| Metric | Score | Interpretation |
| :--- | :--- | :--- |
| **Accuracy (Micro)** | 91% | The model gets the vast majority of *total images* right (likely just guessing the most common rocks). |
| **Precision (Macro)** | 68% | When it predicts a specific rock type, it is correct about 68% of the time, averaged across all types. |
| **Recall (Macro)** | 61% | It successfully finds 61% of the actual instances of a given rock type, averaged across all types. |
| **F1 Score (Macro)** | **63%** | **The most informative metric here.** The large gap between the 91% Accuracy and 63% Macro F1 proves the model is struggling significantly with the minority rock classes. |

## 3.3 Combine both step and train function

#### Packaging Training and Evaluation into a `train()` Function

Now we need a way to put our `train_step()` and `test_step()` functions together. To do so, we'll package them up in a `train()` function. This function will handle both training the model and evaluating it over a specified number of epochs.

##### Function Workflow
The `train()` function will perform the following steps:
1. **Accept Parameters**: Takes in a model, a training `DataLoader`, a test `DataLoader`, an optimizer, a loss function, and the number of epochs to run.
2. **Initialize Results Tracker**: Creates an empty results dictionary with keys for `train_loss`, `train_acc`, `test_loss`, and `test_acc`. We will populate these lists as training progresses.
3. **Execute Epoch Loop**: Loops through the training and test step functions for the specified number of epochs.
4. **Log Progress**: Prints out metrics showing what is happening at the end of each epoch.
5. **Update and Return Metrics**: Updates the results dictionary with the latest metrics every epoch and returns the filled dictionary.

To keep track of the number of epochs we've been through, we will import `tqdm` from `tqdm.auto`. `tqdm` is one of the most popular progress bar libraries for Python, and `tqdm.auto` automatically decides what kind of progress bar is best for your computing environment (e.g., Jupyter Notebook vs. a standard Python script).


# 4. Models

```text
Models trained from scratch
│
├── prepare_classification_data()
│       │
│       └── create_transformation()
│
│       You control:
│       ├── IMG_SIZE
│       ├── resize
│       ├── mean/std
│       └── augmentation
│
└────────────────────────────────


Pretrained models
│
├── prepare_pretrained_classification_data()
│       │
│       └── create_pretrained_transformation()
│
│       Weights control:
│       ├── crop size
│       ├── resize size
│       ├── mean/std
│       ├── interpolation
│       └── antialias
│
│       You control:
│       ├── remove_bottom_annotation
│       ├── horizontal flip
│       ├── vertical flip
│       └── ColorJitter
```


```text
EfficientNet-B0
      ↓
traditional efficient CNN

ConvNeXt-Tiny
      ↓
modern CNN

Swin-T
      ↓
hierarchical transformer

DINOv2 ViT-S/14
      ↓
pure ViT + modern self-supervised pretraining
```

## 4.0 model_0_0_tinyVGG 

In [ ]:
from modular.model_builder import TinyVGG
torch.manual_seed(42)
model_0_0_tinyVGG = TinyVGG(input_shape=3, hidden_units=10, output_shape=len(class_names)).to(device)
model_0_0_tinyVGG

#### Try a forward pass on a single image (to test the model)

In [ ]:
image_batch, label_batch =next(iter(train_dataloader_5))

In [ ]:
model_0_0_tinyVGG(image_batch.to(device))

#### Use torchinfo to get an idea of the shapes going through our model

In [ ]:
from torchinfo import summary
summary(model_0_0_tinyVGG, input_size=[1, 3, 224, 224]) # do a test pass through of an example input siz

## 4.1 Import Meta Models

In [ ]:
dinov2_vits14 = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

In [ ]:
from modular.model_builder import DINOv2Classifier
model_dinov2 = DINOv2Classifier(output_shape=len(class_names)).to(device)

In [ ]:
model_dinov2

#### Check that all parameters are trainable

In [ ]:
trainable_params = sum(
    p.numel()
    for p in model_dinov2.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model_dinov2.parameters()
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# transformations
from torchvision import transforms
from modular.data_setup import remove_bottom_annotation
dinov2_train_transform = transforms.Compose([
    transforms.Lambda(lambda img: remove_bottom_annotation(img, pixels=90)),

    transforms.RandomResizedCrop(
        224,
        interpolation=transforms.InterpolationMode.BICUBIC
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dinov2_val_transform = transforms.Compose([
    transforms.Lambda(lambda img: remove_bottom_annotation(img, pixels=90)),

    transforms.Resize(
        256,
        interpolation=transforms.InterpolationMode.BICUBIC
    ),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# check to make sure its working
image_batch, label_batch =next(iter(train_dataloader_5))
model_dinov2(image_batch.to(device))

In [ ]:
# check to make sure its working
dummy_batch = torch.randn(32, 3, 224, 224).to(device)

with torch.no_grad():
    output = model_dinov2(dummy_batch)

print("Input shape :", dummy_batch.shape)
print("Output shape:", output.shape)

## 4.2 import all pretained models from torch vision

In [ ]:
from torchvision.models import (
    # EfficientNet
    efficientnet_b0,
    EfficientNet_B0_Weights,

    efficientnet_b1,
    EfficientNet_B1_Weights,

    efficientnet_b3,
    EfficientNet_B3_Weights,

    # DenseNet
    densenet121,
    DenseNet121_Weights,

    densenet169,
    DenseNet169_Weights,

    # ResNet
    resnet18,
    ResNet18_Weights,

    resnet50,
    ResNet50_Weights,

    # MobileNet
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,

    # ConvNeXt
    convnext_tiny,
    ConvNeXt_Tiny_Weights,

    convnext_small,
    ConvNeXt_Small_Weights,

    # Swin Transformer
    swin_t,
    Swin_T_Weights,
)

# meta models
dinov2_vits14 = torch.hub.load("facebookresearch/dinov2","dinov2_vits14")

In [ ]:
efficientnet_b0

In [ ]:
EfficientNet_B0_Weights

## 4.3 The function to build all other models 
Rather than creating all these models one by one, I will make a function to contain all my pretrain models

In [ ]:
import torch
import torchvision
from torch import nn


def create_pretrained_model(model_name: str,output_shape: int,freeze_backbone: bool = True):

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # EfficientNet-B0
    # ---------------------------------------------------------
    if model_name == "efficientnet_b0":
        model = torchvision.models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features,output_shape)

    # ---------------------------------------------------------
    # EfficientNet-B1
    # ---------------------------------------------------------
    elif model_name == "efficientnet_b1":
        model = torchvision.models.efficientnet_b1(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features,output_shape)

    # ---------------------------------------------------------
    # EfficientNet-B3
    # ---------------------------------------------------------
    elif model_name == "efficientnet_b3":
        model = torchvision.models.efficientnet_b3(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features,output_shape)

    # ---------------------------------------------------------
    # DenseNet121
    # ---------------------------------------------------------
    elif model_name == "densenet121":
        model = torchvision.models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features,output_shape)

    # ---------------------------------------------------------
    # DenseNet169
    # ---------------------------------------------------------
    elif model_name == "densenet169":
        model = torchvision.models.densenet169(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features,output_shape)

    # ---------------------------------------------------------
    # MobileNet-V3-Large
    # ---------------------------------------------------------
    elif model_name == "mobilenet_v3_large":
        model = torchvision.models.mobilenet_v3_large(weights=weights)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features,output_shape)

    # ---------------------------------------------------------
    # ResNet18
    # ---------------------------------------------------------
    elif model_name == "resnet18":
        model = torchvision.models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features,output_shape)

    # ---------------------------------------------------------
    # ResNet50
    # ---------------------------------------------------------
    elif model_name == "resnet50":
        model = torchvision.models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features,output_shape)

    # ---------------------------------------------------------
    # ConvNeXt-Tiny
    # ---------------------------------------------------------
    elif model_name == "convnext_tiny":
        model = torchvision.models.convnext_tiny(weights=weights)

        model.classifier[2] = nn.Linear(model.classifier[2].in_features,output_shape)

    # ---------------------------------------------------------
    # ConvNeXt-Small
    # ---------------------------------------------------------
    elif model_name == "convnext_small":
        model = torchvision.models.convnext_small(weights=weights)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features,output_shape)

    # ---------------------------------------------------------
    # Swin-T
    # ---------------------------------------------------------
    elif model_name == "swin_t":
        model = torchvision.models.swin_t(weights=weights)
        model.head = nn.Linear(model.head.in_features,output_shape)
        
    # ---------------------------------------------------------
    # dinov2_vits14
    # ---------------------------------------------------------
    elif model_name == "dinov2_vits14":
        #backbone = torch.hub.load("facebookresearch/dinov2","dinov2_vits14",pretrained=False)
        model = DINOv2Classifier(output_shape=output_shape,freeze_backbone=freeze_backbone)
    

    else:
        raise ValueError(
            f"Unsupported model_name: {model_name}"
        )

    return model

In [ ]:
# weights helper function 
def get_pretrained_weights(model_name: str):

    if model_name == "efficientnet_b0":
        return torchvision.models.EfficientNet_B0_Weights.DEFAULT

    elif model_name == "efficientnet_b1":
        return torchvision.models.EfficientNet_B1_Weights.DEFAULT
        
    elif model_name == "efficientnet_b3":
        return torchvision.models.EfficientNet_B3_Weights.DEFAULT
        
    elif model_name == "densenet121":
        return torchvision.models.DenseNet121_Weights.DEFAULT
        
    elif model_name == "densenet169":
        return torchvision.models.DenseNet169_Weights.DEFAULT

    elif model_name == "mobilenet_v3_large":
        return torchvision.models.MobileNet_V3_Large_Weights.DEFAULT

    elif model_name == "resnet18":
        return torchvision.models.ResNet18_Weights.DEFAULT
    
    elif model_name == "resnet50":
        return torchvision.models.ResNet50_Weights.DEFAULT

    elif model_name == "convnext_tiny":
        return torchvision.models.ConvNeXt_Tiny_Weights.DEFAULT

    elif model_name == "convnext_small":
        return torchvision.models.ConvNeXt_Small_Weights.DEFAULT

    elif model_name == "swin_t":
        return torchvision.models.Swin_T_Weights.DEFAULT
        

    elif model_name == "dinov2_vits14":
        return None


    else:
        raise ValueError(
            f"Unsupported model_name: {model_name}"
        )

In [ ]:
# torch info for model summary function # model_summary() cannot handle DINOv2
from torchinfo import summary
def model_summary(
    model,
    weights,
    batch_size=32
):
    """
    Print a TorchInfo summary using the input size required
    by the pretrained weights.
    """
    # Get preprocessing information from pretrained weights
    preprocessing = weights.transforms()
    print(f"Pretrained transformation:\n{preprocessing}\n")
    crop_size = preprocessing.crop_size
    # Determine model input height and width
    if len(crop_size) == 1:
        height = width = crop_size[0]
    else:
        height, width = crop_size

    # Print model summary
    model_stats = summary(
        model=model,
        input_size=(batch_size, 3, height, width),

        col_names=[
            "input_size",
            "output_size",
            "num_params",
            "trainable"
        ],

        col_width=20,
        row_settings=["var_names"],

        # IMPORTANT
        verbose=1
    )

    return model_stats

# 5 Experiments

In [ ]:
from modular.engine import run_experiment
from modular.train_test import train_test
import numpy as np
from modular.data_setup import prepare_pretrained_classification_data
pretrained_model_experiments = {}

## 5.0 model_0_1_tinyVGG on 5% of the data set. No Augmentation

| Class       | Full train | 5% subset |
| ----------- | ---------: | --------: |
| class4      |         61 |         3 |
| **class5**  |      **8** |     **0** |
| class8      |        102 |         5 |
| class11     |        144 |         7 |
| class14     |        144 |         7 |
| **class15** |      **8** |     **0** |
| class16     |         41 |         2 |
| class17     |         82 |         4 |

Above is the issue with the class imbalance. there are some classes that have no pictures in them. this might cause issue doing the experimentation plan
experiment heavily on 5% → take winners to 20% → then full dataset

### 5.0.0 Create transform and dataloader with no data augmentation

In [ ]:
from torchvision import transforms
IMG_SIZE=224
simple_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor() # use ToTensor() last to get everything between 0 & 1
])


In [ ]:
train_subset_5_dataset

In [ ]:
subset_indices_5 = train_subset_5_dataset.indices

In [ ]:
# Full dataset, but with simple transform
train_dataset_simple = RockClassificationDataset(
    root_dir=train_dir,
    image_transform=simple_transform
)

# Apply the SAME 5% indices only to the train
train_subset_5_simple = torch.utils.data.Subset(
    train_dataset_simple,
    subset_indices_5
)



val_dataset_5_simple = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=simple_transform
)

In [ ]:
BATCH_SIZE = 32

# Turn datasets into iterables (batches)
train_dataloader_5_simple = DataLoader(train_subset_5_simple, 
    batch_size=BATCH_SIZE, 
    num_workers=os.cpu_count(),                          
    shuffle=True 
)
val_dataloader_5_simple = DataLoader(val_dataset_5_simple, 
    batch_size=BATCH_SIZE, 
    num_workers=os.cpu_count(),                           
    shuffle=False 
)

In [ ]:
torch.manual_seed(42)
model_0_0_tinyVGG_results = TinyVGG(input_shape=3, 
    hidden_units=10, 
    output_shape=len(class_names)).to(device)
model_0_0_tinyVGG_results

In [ ]:
from modular.engine import run_experiment
from modular.train_test import train_test
from torch import nn
model_0_0_tinyVGG, model_0_0_tinyVGG_results, training_time = run_experiment(
    model_class=TinyVGG,

    model_kwargs={
        "input_shape": 3,
        "hidden_units": 10,
        "output_shape": len(class_names),
        "padding": 0,
        "dropout": 0.0
    },

    train_dataloader=train_dataloader_5_simple,
    val_dataloader=val_dataloader_5_simple,

    train_metrics=train_metrics,
    test_metrics=test_metrics,
    class_names=class_names,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={"lr": 0.001},

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={},
    run_name="tinyvgg_baseline_no_augmentation",
    seed=42
)

## 5.1 model_0_1_tinyVGG on 5% of the data Augmentation

This section might be out of order because I used the transformation that was developed in section 1. that is why u don't see the creation of the transformation and the dataloaders in here. Also for all other experiments that uses this augmentation, you are not going to see it

### 5.1.1 Create train & test loop functions

In [ ]:
loss_fn = nn.CrossEntropyLoss() # this is also called "criterion"/"cost function" in some places
optimizer = torch.optim.SGD(params=model_0_0_tinyVGG.parameters(), lr=0.1)

In [ ]:
model_0_1_tinyVGG, model_0_1_tinyVGG_results, training_time = run_experiment(
    model_class=TinyVGG,

    model_kwargs={
        "input_shape": 3,
        "hidden_units": 10,
        "output_shape": len(class_names),
        "padding": 0,
        "dropout": 0.0
    },

    train_dataloader=train_dataloader_5,
    val_dataloader=val_dataloader_5,

    train_metrics=train_metrics,
    test_metrics=test_metrics,
    class_names=class_names,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={"lr": 0.001},

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={},
    run_name="tinyvgg_augmentation",
    seed=42
)

## 5.2 model_0_2_tinyVGG on 5% of the data Augmentation experimentting with image size
See whether you retain performance with much cheaper training

### experimenting for the best image size

In [ ]:
from modular.data_setup import prepare_classification_data

In [ ]:
image_sizes =   [80,160,244,320,640]

image_size_experiments = {}

for img_size in image_sizes:
    resize = round(img_size * 256 / 224) # to give a much much fairer image-resolution experiment.

    print("\n" + "=" * 60)
    print(f"IMAGE SIZE EXPERIMENT: {img_size} x {img_size}")
    print("=" * 60)

    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = prepare_classification_data(
        train_dataset=train_subset_5_dataset,
        val_dir=val_dir,
        IMG_SIZE=img_size,
        resize=resize,
        normalization="mydataset",
        pixels=90,
        batch_size=32,
        num_workers=4,
        n=5,
        seed=42
    )

    # ---------------------------------------------------------
    # 2. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=TinyVGG,

        model_kwargs={
            "input_shape": 3,
            "hidden_units": 10,
            "output_shape": len(class_names),
            "padding": 0,
            "dropout": 0.0
        },

        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=5,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.001},

        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={},

        run_name=f"tinyvgg_augmentation_imgsize_{img_size}",

        seed=42
    )

    # ---------------------------------------------------------
    # 3. Store experiment
    # ---------------------------------------------------------
    image_size_experiments[img_size] = {
        "model": model,
        "results": results,
        "training_time": training_time
    }

## 5.3  model_0_3_tinyVGG on 5% of the data Augmentation Weighted CE

In [ ]:
import torch
import numpy as np

# Get labels from the exact training subset
labels = [
    train_subset_5_dataset.dataset.samples[i][1]
    for i in train_subset_5_dataset.indices
]

class_counts = np.bincount(
    labels,
    minlength=len(class_names)
)

class_weights = np.zeros(
    len(class_counts),
    dtype=np.float32
)

# Only calculate weights for classes that are present
present_classes = class_counts > 0

class_weights[present_classes] = (
    1 / np.sqrt(class_counts[present_classes])
)

# Normalize only the non-zero weights
class_weights[present_classes] /= (
    class_weights[present_classes].mean()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

In [ ]:
image_sizes =   [80]



for img_size in image_sizes:
    resize = round(img_size * 256 / 224) # to give a much much fairer image-resolution experiment.

    print("\n" + "=" * 60)
    print(f"IMAGE SIZE EXPERIMENT: {img_size} x {img_size}")
    print("=" * 60)

    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = prepare_classification_data(
        train_dataset=train_subset_5_dataset,
        val_dir=val_dir,
        IMG_SIZE=img_size,
        resize=resize,
        normalization="mydataset",
        pixels=90,
        batch_size=32,
        num_workers=4,
        n=5,
        seed=42
    )

    # ---------------------------------------------------------
    # 2. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=TinyVGG,

        model_kwargs={
            "input_shape": 3,
            "hidden_units": 10,
            "output_shape": len(class_names),
            "padding": 0,
            "dropout": 0.0
        },

        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=5,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.001},

        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={"weight": class_weights},

        run_name="tinyvgg_img80_weightedCE_sqrt",

        seed=42
    )

    # ---------------------------------------------------------
    # 3. Store experiment
    # ---------------------------------------------------------
    image_size_experiments[img_size] = {
        "model": model,
        "results": results,
        "training_time": training_time
    }

## 5.4 model_0_4_efficientnet_b0_on 5% of the data Augmentation

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name = "efficientnet_b0"
weights = get_pretrained_weights(model_name)

# ---------------------------------------------------------
# 1. Prepare data
# ---------------------------------------------------------
figs, train_dataloader, val_dataloader = (
    prepare_pretrained_classification_data(
        train_dataset=train_subset_5_dataset,
        val_dir=val_dir,
        weights=weights,

        pixels=90,

        RandomHorizontalFlip_p=0.5,
        RandomVerticalFlip_p=0.0,
        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.03,

        batch_size=32,
        num_workers=4,

        n=5,
        seed=42
    )
)

# ---------------------------------------------------------
# 2. Create temporary model and print summary using the function created from torch info
# ---------------------------------------------------------
model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
model_check = model_check.to(device)
model_summary(model=model_check,weights=weights)
del model_check

# ---------------------------------------------------------
# 3. Train model
# ---------------------------------------------------------
model, results, training_time = run_experiment(
    model_class=create_pretrained_model,

    model_kwargs={
        "model_name": model_name,
        "output_shape": len(class_names)
    },

    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,

    train_metrics=train_metrics,
    test_metrics=test_metrics,
    class_names=class_names,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={
        "lr": 0.0001
    },

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={},

    run_name=f"{model_name}_fullfinetune_5percent",

    seed=42
)


# ---------------------------------------------------------
# 4. Store experiment
# ---------------------------------------------------------

pretrained_model_experiments[model_name] = {
    "model": model,
    "results": results,
    "training_time": training_time,
    "weights": weights
}


## 5.5 model_0_5_resnet18 5% of the data Augmentation

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name = "resnet18"
weights = get_pretrained_weights(model_name)

# ---------------------------------------------------------
# 1. Prepare data
# ---------------------------------------------------------
figs, train_dataloader, val_dataloader = (
    prepare_pretrained_classification_data(
        train_dataset=train_subset_5_dataset,
        val_dir=val_dir,
        weights=weights,

        pixels=90,

        RandomHorizontalFlip_p=0.5,
        RandomVerticalFlip_p=0.0,
        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.03,

        batch_size=32,
        num_workers=4,

        n=5,
        seed=42
    )
)

# ---------------------------------------------------------
# 2. Create temporary model and print summary using the function created from torch info
# ---------------------------------------------------------
model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
model_check = model_check.to(device)
model_summary(model=model_check,weights=weights)
del model_check

# ---------------------------------------------------------
# 3. Train model
# ---------------------------------------------------------
model, results, training_time = run_experiment(
    model_class=create_pretrained_model,

    model_kwargs={
        "model_name": model_name,
        "output_shape": len(class_names)
    },

    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,

    train_metrics=train_metrics,
    test_metrics=test_metrics,
    class_names=class_names,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={
        "lr": 0.0001
    },

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={},

    run_name=f"{model_name}_fullfinetune_5percent",

    seed=42
)


# ---------------------------------------------------------
# 4. Store experiment
# ---------------------------------------------------------

pretrained_model_experiments[model_name] = {
    "model": model,
    "results": results,
    "training_time": training_time,
    "weights": weights
}

## 5.6 model_0_6_convnext_tiny 5% of the data Augmentation

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name = "convnext_tiny"
weights = get_pretrained_weights(model_name)

# ---------------------------------------------------------
# 1. Prepare data
# ---------------------------------------------------------
figs, train_dataloader, val_dataloader = (
    prepare_pretrained_classification_data(
        train_dataset=train_subset_5_dataset,
        val_dir=val_dir,
        weights=weights,

        pixels=90,

        RandomHorizontalFlip_p=0.5,
        RandomVerticalFlip_p=0.0,
        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.03,

        batch_size=32,
        num_workers=4,

        n=5,
        seed=42
    )
)

# ---------------------------------------------------------
# 2. Create temporary model and print summary using the function created from torch info
# ---------------------------------------------------------
model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
model_check = model_check.to(device)
model_summary(model=model_check,weights=weights)
del model_check

# ---------------------------------------------------------
# 3. Train model
# ---------------------------------------------------------
model, results, training_time = run_experiment(
    model_class=create_pretrained_model,

    model_kwargs={
        "model_name": model_name,
        "output_shape": len(class_names)
    },

    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,

    train_metrics=train_metrics,
    test_metrics=test_metrics,
    class_names=class_names,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={
        "lr": 0.0001
    },

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={},

    run_name=f"{model_name}_fullfinetune_5percent",

    seed=42
)


# ---------------------------------------------------------
# 4. Store experiment
# ---------------------------------------------------------

pretrained_model_experiments[model_name] = {
    "model": model,
    "results": results,
    "training_time": training_time,
    "weights": weights
}

## 5.7 model_0_6_experiment 20% of the data Augmentation with different models

In [ ]:

# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["efficientnet_b0","efficientnet_b3","densenet121","densenet169","mobilenet_v3_large","resnet18","resnet50","convnext_tiny","convnext_small","swin_t"]


for model_name in model_name_list:
    
    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)
    
    
    weights = get_pretrained_weights(model_name)
    
    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,
    
            pixels=90,
    
            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,
    
            batch_size=32,
            num_workers=4,
    
            n=5,
            seed=42
        )
    )
    
    # ---------------------------------------------------------
    # 2. Create temporary model and print summary using the function created from torch info
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    
    # ---------------------------------------------------------
    # 3. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,
    
        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
    
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
    
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,
    
        device=device,
        epochs=5,
    
        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={
            "lr": 0.0001
        },
    
        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={},
    
        run_name=f"{model_name}_fullfinetune_20percent",
    
        seed=42
    )
    
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    
    pretrained_model_experiments[model_name] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.8 model_0_8_experiment 20% of the data Augmentation with smaller different models trained for 10 epochs

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny","convnext_small","swin_t", "densenet169"]


for model_name in model_name_list:
    
    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)
    
    
    weights = get_pretrained_weights(model_name)
    
    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,
    
            pixels=90,
    
            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,
    
            batch_size=32,
            num_workers=4,
    
            n=3,
            seed=42
        )
    )
    
    # ---------------------------------------------------------
    # 2. Create temporary model and print summary using the function created from torch info
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    
    # ---------------------------------------------------------
    # 3. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,
    
        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
    
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
    
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,
    
        device=device,
        epochs=10,
    
        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={
            "lr": 0.0001
        },
    
        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={},
    
        run_name=f"{model_name}_fullfinetune_20percent_10epochs",
    
        seed=42
    )
    
    # Training finishes above
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    
    pretrained_model_experiments[model_name] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.9 model_0_9_experiment 20% of the data Augmentation convnext_tiny Weighted CE
I am using weighted ce to handle the Handle imbalance

In [ ]:
#First, calculate class weights from the 20% training subset only

train_labels_20 = [
    train_subset_20_dataset[i][1]
    for i in range(len(train_subset_20_dataset))
]

class_counts = np.bincount(
    train_labels_20,
    minlength=len(class_names)
)

print("Class counts:", class_counts)

In [ ]:
#Check first that every class exists
if np.any(class_counts == 0):
    missing_classes = np.where(class_counts == 0)[0]

    raise ValueError(
        f"Missing classes in 20% training subset: "
        f"{missing_classes.tolist()}"
    )

In [ ]:
# square-root inverse-frequency weighting
class_weights = 1 / np.sqrt(class_counts)

# Normalize so average weight is approximately 1
class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(class_weights,dtype=torch.float32,device=device)

print("Class weights:", class_weights)

In [ ]:
loss_class = nn.CrossEntropyLoss

loss_kwargs = {
    "weight": class_weights
}

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:
    
    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)
    
    
    weights = get_pretrained_weights(model_name)
    
    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,
    
            pixels=90,
    
            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,
    
            batch_size=32,
            num_workers=4,
    
            n=3,
            seed=42
        )
    )
    
    # ---------------------------------------------------------
    # 2. Create temporary model and print summary using the function created from torch info
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 3. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,
    
        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
    
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
    
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,
    
        device=device,
        epochs=10,
    
        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={
            "lr": 0.0001
        },
    
        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={"weight": class_weights},
    
        run_name=f"{model_name}_fullfinetune_20percent_CE_Weigthed",
    
        seed=42
    )
    
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    
    pretrained_model_experiments[model_name] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.10 model_0_10_experiment 20% of the data Augmentation convnext_tiny WeightedRandomSampler
TO Compensate for oversampling <br>
For comparison: training metrics now describe the resampled images, so compare experiments primarily using validation macro F1 and per-class recall.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare data with ConvNeXt pretrained transforms
    # ---------------------------------------------------------
    figs, train_dataloader_normal, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )

    # Get the TRANSFORMED training dataset
    train_dataset_transformed = train_dataloader_normal.dataset
    # ---------------------------------------------------------
    # 2. Calculate weighted sampling probabilities
    # ---------------------------------------------------------
    class_sampling_weights = 1 / np.sqrt(class_counts)
    sample_weights = [class_sampling_weights[label]for label in train_labels_20]
    sample_weights = torch.tensor(sample_weights,dtype=torch.double )
    # ---------------------------------------------------------
    # 3. Create weighted sampler
    # ---------------------------------------------------------
    weighted_sampler_20 = WeightedRandomSampler(weights=sample_weights,num_samples=len(sample_weights),replacement=True)
    # ---------------------------------------------------------
    # 4. Create weighted training DataLoader
    # ---------------------------------------------------------
    train_dataloader_20_sqrt_sampler = DataLoader(
        train_dataset_transformed,
        batch_size=32,
        sampler=weighted_sampler_20,
        num_workers=4,
        pin_memory=True
    )
    # ---------------------------------------------------------
    # 5. Model summary
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 6. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,

        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },

        train_dataloader=train_dataloader_20_sqrt_sampler,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=10,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={
            "lr": 0.0001
        },

        # NORMAL CE — weighting is being done by sampler
        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={},

        run_name=f"{model_name}_fullfinetune_20percent_weighted_sampler",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 7. Store experiment
    # ---------------------------------------------------------

    pretrained_model_experiments[model_name] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.10a model_0_10_experiment 20% of the data Augmentation convnext_tiny WeightedRandomSampler more aggresive sampler
TO Compensate for oversampling <br>
For comparison: training metrics now describe the resampled images, so compare experiments primarily using validation macro F1 and per-class recall.<br>
1 / class_counts for more aggressive sampling

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare data with ConvNeXt pretrained transforms
    # ---------------------------------------------------------
    figs, train_dataloader_normal, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )

    # Get the TRANSFORMED training dataset
    train_dataset_transformed = train_dataloader_normal.dataset
    # ---------------------------------------------------------
    # 2. Calculate weighted sampling probabilities
    # ---------------------------------------------------------
    class_sampling_weights = 1 / class_counts
    sample_weights = [class_sampling_weights[label]for label in train_labels_20]
    sample_weights = torch.tensor(sample_weights,dtype=torch.double )
    # ---------------------------------------------------------
    # 3. Create weighted sampler
    # ---------------------------------------------------------
    weighted_sampler_20 = WeightedRandomSampler(weights=sample_weights,num_samples=len(sample_weights),replacement=True)
    # ---------------------------------------------------------
    # 4. Create weighted training DataLoader
    # ---------------------------------------------------------
    train_dataloader_20_weighted_sampler = DataLoader(
        train_dataset_transformed,
        batch_size=32,
        sampler=weighted_sampler_20,
        num_workers=4,
        pin_memory=True
    )
    # ---------------------------------------------------------
    # 5. Model summary
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 6. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,

        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },

        train_dataloader=train_dataloader_20_weighted_sampler,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=10,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={
            "lr": 0.0001
        },

        # NORMAL CE — weighting is being done by sampler
        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={},

        run_name=f"{model_name}_fullfinetune_20percent_weighted_sampler_inverse",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 7. Store experiment
    # ---------------------------------------------------------

    pretrained_model_experiments[f"{model_name}_sampler_inverse"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.11 model_0_11_experiment 20% of the data Augmentation convnext_tiny Focal loss
Handle difficult/rare classes

In [ ]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("weight", weight)

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()

        loss = -((1 - pt) ** self.gamma) * log_pt

        if self.weight is not None:
            sample_weights = self.weight[targets]
            loss = sample_weights * loss
            return loss.sum() / sample_weights.sum().clamp_min(1e-12)

        return loss.mean()

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare NORMAL training data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 3. Train model with Focal Loss
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,

        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=7,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.0001},
        # -----------------------------
        # FOCAL LOSS
        # -----------------------------
        loss_class=FocalLoss,
        loss_kwargs={"gamma": 2.0,"weight": None},

        run_name=f"{model_name}_fullfinetune_20percent_focal_loss_2",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments[f"{model_name}_focal_gamma2"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.12 model_0_12_experiment 20% of the data Augmentation convnext_tiny Class-weighted focal loss  best so far
Conceptually, it combines:

class weights → rare classes matter more <br>
focal term → hard examples matter more<br>

So a difficult example from a rare class gets emphasized by both mechanisms. <br>
Best 20% imbalance strategy: <br>
ConvNeXt-Tiny + Weighted Focal Loss <br>
gamma = 2 <br>
class weights = inverse sqrt frequency <br>
sampling = normal <br>

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare NORMAL training data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 3. Train model with Focal Loss
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,

        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=10,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.0001},
        # -----------------------------
        # FOCAL LOSS
        # -----------------------------
        loss_class=FocalLoss,
        loss_kwargs={"gamma": 2.0,"weight": class_weights},

        run_name=f"{model_name}_fullfinetune_20percent_focal_loss_2_weighted_CE",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments[f"{model_name}_weighted_focal_gamma2"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.12a model_0_12_experiment 20% of the data Augmentation convnext_tiny Class-weighted focal loss weight=NONE

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare NORMAL training data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    model_check = torchvision.models.convnext_tiny(weights=None,num_classes=len(class_names)).to(device)

    # Uses preprocessing information to determine the input size.
    model_summary(model=model_check, weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 3. Train model with Focal Loss
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=torchvision.models.convnext_tiny,

        model_kwargs={
            "weights": None,
            "num_classes": len(class_names)
        },
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=10,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.0001},
        # -----------------------------
        # FOCAL LOSS
        # -----------------------------
        loss_class=FocalLoss,
        loss_kwargs={"gamma": 2.0,"weight": class_weights},

        run_name=f"{model_name}_fromscratch_20percent_focal_loss_2_weighted_CE",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments["convnext_tiny_fromscratch_weighted_focal_gamma2"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": None
    }

## 5.13 model_0_12_experiment 20% of the data Augmentation convnext_tiny focal loss with weighted random sampling
sampling is 1/sqrt(class)

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare NORMAL training data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_subset_20_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)
    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 3. Train model with Focal Loss
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,

        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
        train_dataloader=train_dataloader_20_sqrt_sampler,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=7,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.0001},
        # -----------------------------
        # FOCAL LOSS
        # -----------------------------
        loss_class=FocalLoss,
        loss_kwargs={"gamma": 2.0,"weight": None},

        run_name=f"{model_name}_fullfinetune_20percent_focal_loss_weighted_sampler",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments[f"{model_name}_focal_sqrt_sampler"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 5.14 model_0_14_experiment 20% of the data with augmentation: DINOv2 full fine-tuning, inverse-frequency sampler and focal loss

Uses pretrained DINOv2 weights with all layers trainable, random resized crops and horizontal flips. Sampling uses inverse class frequency; focal loss uses gamma=2 without class weights.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_dataset_dino = RockClassificationDataset(
    root_dir=train_dir, image_transform=dinov2_train_transform
)
# Reuse the same images as the existing 20% training subset.
assert train_dataset_dino.samples == train_subset_20_dataset.dataset.samples
train_subset_20_dino = torch.utils.data.Subset(
    train_dataset_dino, train_subset_20_dataset.indices
)

# Calculate inverse-frequency sampling weights in this subset's order.
train_labels_dino_20 = np.asarray([
    train_dataset_dino.samples[i][1]
    for i in train_subset_20_dino.indices
])
class_counts_dino_20 = np.bincount(
    train_labels_dino_20, minlength=len(class_names)
)
sample_weights_dino_20 = torch.tensor(
    1.0 / class_counts_dino_20[train_labels_dino_20], dtype=torch.double
)
weighted_sampler_dino_20 = WeightedRandomSampler(
    weights=sample_weights_dino_20,
    num_samples=len(train_subset_20_dino),
    replacement=True
)
print("DINO subset:", len(train_subset_20_dino))
print("Sampler: inverse class frequency")
print("Sampler weights:", len(weighted_sampler_dino_20.weights))

In [ ]:
val_dataset_dino = RockClassificationDataset(
    root_dir=val_dir, image_transform=dinov2_val_transform
)
print("Training subset size:", len(train_subset_20_dino))
print("Sampler weights:", len(weighted_sampler_dino_20.weights))
print("Validation size:", len(val_dataset_dino))

assert train_dataset_dino.classes == class_names
assert train_dataset_dino.class_to_idx == val_dataset_dino.class_to_idx
assert len(train_subset_20_dino) == len(weighted_sampler_dino_20.weights)

In [ ]:
from modular.data_setup import remove_bottom_annotation
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["dinov2_vits14"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)
    # ---------------------------------------------------------
    # 0. Get weights
    # ---------------------------------------------------------
    weights = get_pretrained_weights(model_name)
    # No torchvision weights enum; DINOv2Classifier loads pretrained=True.

    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    if model_name == "dinov2_vits14":

  
        # Use the datasets and inverse-frequency sampler checked above.
        train_dataloader = DataLoader(
            train_subset_20_dino,
            batch_size=32,
            sampler=weighted_sampler_dino_20,
            num_workers=4
        )
        val_dataloader = DataLoader(
            val_dataset_dino,
            batch_size=32,
            shuffle=False,
            num_workers=4
        )

    else:
        figs, train_dataloader, val_dataloader = (
            prepare_pretrained_classification_data(
                train_dataset=train_subset_20_dataset,
                val_dir=val_dir,
                weights=weights,

                pixels=90,

                RandomHorizontalFlip_p=0.5,
                RandomVerticalFlip_p=0.0,
                brightness=0.15,
                contrast=0.15,
                saturation=0.1,
                hue=0.03,

                batch_size=32,
                num_workers=4,

                n=3,
                seed=42
            )
        )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    #model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names)).to(device)
    #model_summary(model=model_check,weights=weights)

    # ---------------------------------------------------------
    # 3. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,
        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names),
            "freeze_backbone": False
        },

        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=30,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.0001},

        # -----------------------------
        # FOCAL LOSS
        # -----------------------------
        loss_class=FocalLoss,
        loss_kwargs={"gamma": 2.0,"weight": None},
        run_name=(f"{model_name}_fullfinetune_20percent_focal_gamma2_inverse_sampler_30epochs"),
        seed=42
    )
    model = model.cpu()

    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments[f"{model_name}_fullfinetune_focal_gamma2_inverse_sampler"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights,  # No torchvision enum for DINOv2.
        "pretrained": True,
        "freeze_backbone": False,
        "sampling": "inverse_frequency"
    }

## 5.15 model_0_14_experiment 20% of the with Augmentation dinov2_vits14 with CE Loss second best

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["dinov2_vits14"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)
    # ---------------------------------------------------------
    # 0. Get weights
    # ---------------------------------------------------------
    weights = get_pretrained_weights(model_name)
    # DINOv2 -> None

    # ---------------------------------------------------------
    # 1. Prepare data
    # ---------------------------------------------------------
    if model_name == "dinov2_vits14":

  
        # Recreate full training dataset with DINO transform
        train_dataset_dino = RockClassificationDataset(root_dir=train_dir,image_transform=dinov2_train_transform)
        
        # Apply SAME 20% indices
        train_subset_20_dino = torch.utils.data.Subset(train_dataset_dino,train_subset_20_dataset.indices)
        
        # Validation dataset with DINO transform
        val_dataset_dino = RockClassificationDataset(root_dir=val_dir,image_transform=dinov2_val_transform)
        
        # Normal shuffled sampling based on the SAME 20% subset
        train_dataloader = DataLoader(
            train_subset_20_dino,
            batch_size=32,
            #sampler=weighted_sampler_20,
            shuffle=True,
            num_workers=4
        )
        val_dataloader = DataLoader(
            val_dataset_dino,
            batch_size=32,
            shuffle=False,
            num_workers=4
        )

    else:
        figs, train_dataloader, val_dataloader = (
            prepare_pretrained_classification_data(
                train_dataset=train_subset_20_dataset,
                val_dir=val_dir,
                weights=weights,

                pixels=90,

                RandomHorizontalFlip_p=0.5,
                RandomVerticalFlip_p=0.0,
                brightness=0.15,
                contrast=0.15,
                saturation=0.1,
                hue=0.03,

                batch_size=32,
                num_workers=4,

                n=3,
                seed=42
            )
        )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    #model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names)).to(device)
    #model_summary(model=model_check,weights=weights)

    # ---------------------------------------------------------
    # 3. Train model
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,
        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names),
            "freeze_backbone": False
        },

        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=15,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.00001},

        # -----------------------------
        # CROSS ENTROPY
        # -----------------------------
        loss_class=nn.CrossEntropyLoss,
        loss_kwargs={},
        run_name=(f"{model_name}_fullfinetune_20%_CE"),
        seed=42
    )
    model = model.cpu()


    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments[f"{model_name}_fullfinetune_ce_normal"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

# 6. 100 % of training dataset

In [ ]:
train_labels = [label for _, label in train_dataset.samples]

class_counts = np.bincount(
    train_labels,
    minlength=len(class_names)
)

print("Class counts:", class_counts)

class_weights = np.zeros_like(class_counts, dtype=np.float32)

nonzero = class_counts > 0

class_weights[nonzero] = 1 / np.sqrt(class_counts[nonzero])

class_weights[nonzero] /= class_weights[nonzero].mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("Class weights:", class_weights)

In [ ]:
if np.any(class_counts == 0):
    raise ValueError("Some training classes have no samples.")

nonzero = class_counts > 0

class_weights = 1.0 / np.sqrt(class_counts)
class_weights /= class_weights.mean()
class_weights = torch.tensor(
    class_weights, dtype=torch.float32, device=device
)

## 6.0 model_0_12_experiment 100% of the data Augmentation convnext_tiny focal loss Weighted CE best so far. Do this again for 100% of the dataset

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

model_name_list = ["convnext_tiny"]

for model_name in model_name_list:

    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    weights = get_pretrained_weights(model_name)

    # ---------------------------------------------------------
    # 1. Prepare NORMAL training data
    # ---------------------------------------------------------
    figs, train_dataloader, val_dataloader = (
        prepare_pretrained_classification_data(
            train_dataset=train_dataset,
            val_dir=val_dir,
            weights=weights,

            pixels=90,

            RandomHorizontalFlip_p=0.5,
            RandomVerticalFlip_p=0.0,
            brightness=0.15,
            contrast=0.15,
            saturation=0.1,
            hue=0.03,

            batch_size=32,
            num_workers=4,

            n=3,
            seed=42
        )
    )
    # ---------------------------------------------------------
    # 2. Model summary
    # ---------------------------------------------------------
    model_check = create_pretrained_model(model_name=model_name,output_shape=len(class_names))
    model_check = model_check.to(device)

    model_summary(model=model_check,weights=weights)
    del model_check
    # ---------------------------------------------------------
    # 3. Train model with Focal Loss
    # ---------------------------------------------------------
    model, results, training_time = run_experiment(
        model_class=create_pretrained_model,

        model_kwargs={
            "model_name": model_name,
            "output_shape": len(class_names)
        },
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,

        train_metrics=train_metrics,
        test_metrics=test_metrics,
        class_names=class_names,

        device=device,
        epochs=10,

        optimizer_class=torch.optim.Adam,
        optimizer_kwargs={"lr": 0.0001},
        # -----------------------------
        # FOCAL LOSS
        # -----------------------------
        loss_class=FocalLoss,
        loss_kwargs={"gamma": 2.0,"weight": class_weights},

        run_name=f"{model_name}_fullfinetune_100percent_focal_loss_2_weighted_CE_again",

        seed=42
    )
    model = model.cpu()
    # ---------------------------------------------------------
    # 4. Store experiment
    # ---------------------------------------------------------
    pretrained_model_experiments["convnext_tiny_100percent_weighted_focal_gamma2"] = {
        "model": model,
        "results": results,
        "training_time": training_time,
        "weights": weights
    }

## 6.1 model_0_ convnext_tiny Optuna

### 0. clear all memory

In [ ]:
import gc
import torch
import matplotlib.pyplot as plt

# Remove references to previous experiment objects
for name in [
    "model",
    "model_check",
    "best_model",
    "optimizer",
    "loss_fn",
    "results",
]:
    globals().pop(name, None)

# Close matplotlib figures
plt.close("all")

# Python garbage collection
gc.collect()

# Release unused cached GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## 6.2 Second optuna study to test 4 stragegy
**Strategy 1** <br>
Normal sampler <br>
Ordinary CrossEntropyLoss <br>
no class weights <br>

**Strategy 2**<br>
WeightedRandomSampler<br>
Ordinary CrossEntropyLoss<br>
no class weights<br>

**Strategy 3**<br>
Normal sampler<br>
FocalLoss<br>
gamma = 2<br>
weight = class_weights<br>

**Strategy 4** <br>
WeightedRandomSampler<br>
FocalLoss<br>
gamma = 2<br>
weight = class_weights<br>

### 6.2.1. Imports

In [ ]:
from modular.utility import (
    log_confusion_matrix,
    log_f1_by_class,
    log_recall_by_class,
    log_training_images_vs_f1,
    log_confusion_matrix_with_distribution,
)

### 6.2.2 Optimization

In [ ]:
# ============================================================
# Optuna: 4 imbalance strategies
# ConvNeXt-Tiny
# ============================================================

from modular.train_test import train_test
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
import wandb
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall
)

from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

# ============================================================
# 1. Fixed experiment configuration
# ============================================================

MODEL_NAME = "convnext_tiny"

OPTUNA_EPOCHS = 10
N_TRIALS_PER_STRATEGY = 10

BATCH_SIZE = 32
SEED = 42

# Change this name if you want to start a completely fresh set
# of Optuna studies later.
STUDY_TAG = "4strategy_class_merged"
tags=["v2", "optuna"]

STORAGE = "sqlite:///convnext_4strategy_optuna.db"


# ============================================================
# 2. Dataset to use
# ============================================================

# 100% training dataset
TRAIN_DATASET = train_dataset

# If you want to test this first on 20%, use:
# TRAIN_DATASET = train_subset_20_dataset


# ============================================================
# 3. Four strategies
# ============================================================

STRATEGIES = {

    # 1. Normal sampling + weighted cross-entropy.
    "ce_normal": {
        "use_sampler": False,
        "use_focal": False
    },

    # 2. Ordinary CE + WeightedRandomSampler
    "ce_weighted_sampler": {
        "use_sampler": True,
        "use_focal": False
    },

    # 3. Weighted focal loss gamma=2 + normal sampling
    "weighted_focal_normal": {
        "use_sampler": False,
        "use_focal": True
    },

    # 4. Weighted focal loss gamma=2 + WeightedRandomSampler
    "weighted_focal_weighted_sampler": {
        "use_sampler": True,
        "use_focal": True
    }
}


# ============================================================
# 4. Prepare ConvNeXt data ONCE
# ============================================================

weights = get_pretrained_weights(MODEL_NAME)

figs, train_dataloader_normal, val_dataloader = (
    prepare_pretrained_classification_data(

        train_dataset=TRAIN_DATASET,
        val_dir=val_dir,
        weights=weights,

        pixels=90,

        RandomHorizontalFlip_p=0.5,
        RandomVerticalFlip_p=0.0,

        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.03,

        batch_size=BATCH_SIZE
    )
)


# ============================================================
# 5. Helper: extract labels from dataset
# ============================================================

from modular.data_setup import get_dataset_labels


# Get labels from the transformed training dataset
train_labels = get_dataset_labels(train_dataloader_normal.dataset)

train_labels = np.asarray(train_labels,dtype=np.int64)

print("Number of training images:", len(train_labels))
# ============================================================
# 6. Class counts
# ============================================================

class_counts = np.bincount(train_labels,minlength=len(class_names))
print("Class counts:")
print(class_counts)


# ============================================================
# 7. Class weights for WEIGHTED FOCAL LOSS
# ============================================================


print("\nFocal-loss class weights:")
print(class_weights)


# ============================================================
# 8. Sample weights for WeightedRandomSampler
#
#    Here we use standard inverse-frequency sampling:
#
#              1 / class_count
#
#    This is deliberately SEPARATE from the focal-loss
#    class weighting above.
# ============================================================

sampler_class_weights = np.zeros(len(class_names),dtype=np.float64)

sampler_class_weights[nonzero] = (1.0 / class_counts[nonzero])

sample_weights = sampler_class_weights[train_labels]

sample_weights = torch.tensor(sample_weights,dtype=torch.double)

# ============================================================
# 9. Create WeightedRandomSampler DataLoader
# ============================================================

def create_weighted_sampler_dataloader(seed=42):

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator
    )

    loader_kwargs = {
        "dataset": train_dataloader_normal.dataset,
        "batch_size": train_dataloader_normal.batch_size,
        "sampler": sampler,
        "shuffle": False,
        "num_workers": train_dataloader_normal.num_workers,
        "pin_memory": train_dataloader_normal.pin_memory,
        "drop_last": train_dataloader_normal.drop_last,
        "collate_fn": train_dataloader_normal.collate_fn
    }

    # Only use persistent workers when workers actually exist
    if train_dataloader_normal.num_workers > 0:
        loader_kwargs["persistent_workers"] = (
            train_dataloader_normal.persistent_workers
        )

    return DataLoader(**loader_kwargs)


# ============================================================
# 10. Metric helper
# ============================================================
def create_metrics():
    metrics = MetricCollection({
        "accuracy": MulticlassAccuracy(num_classes=len(class_names),average="micro"),
        "f1": MulticlassF1Score(num_classes=len(class_names),average="macro"),
        "precision": MulticlassPrecision(num_classes=len(class_names),average="macro"),
        "recall": MulticlassRecall(num_classes=len(class_names),average="macro")
    })
    train_metrics = metrics.clone(prefix="train_").to(device)
    val_metrics = metrics.clone(prefix="test_").to(device)
    return train_metrics, val_metrics

# ============================================================
# 11. Optuna objective factory
#
#     Makes one objective for each of the 4 strategies.
# ============================================================
def make_objective(strategy_name):
    strategy = STRATEGIES[strategy_name]
    use_sampler = strategy["use_sampler"]
    use_focal = strategy["use_focal"]
    def objective(trial):

        print("\n" + "=" * 75)
        print(
            f"{strategy_name.upper()} | "
            f"OPTUNA TRIAL {trial.number}"
        )
        print("=" * 75)

        # ----------------------------------------------------
        # 1. Clear memory before trial
        # ----------------------------------------------------
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ----------------------------------------------------
        # 2. Hyperparameters
        # ----------------------------------------------------
        lr = trial.suggest_float("lr",1e-5,5e-4,log=True)
        weight_decay = trial.suggest_float("weight_decay",1e-6,5e-2,log=True)
        print(f"Strategy     : {strategy_name}")
        print(f"Learning rate: {lr:.6g}")
        print(f"Weight decay : {weight_decay:.6g}")
        print(f"Sampler      : {use_sampler}")
        print(f"Focal loss   : {use_focal}")
        # ----------------------------------------------------
        # 3. Reproducibility
        # ----------------------------------------------------
        torch.manual_seed(SEED)
        np.random.seed(SEED)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(SEED)
            torch.cuda.manual_seed_all(SEED)
        # ----------------------------------------------------
        # 4. Select training DataLoader
        # ----------------------------------------------------
        if use_sampler:
            train_dataloader = (create_weighted_sampler_dataloader(seed=SEED))
        else:
            train_dataloader = (train_dataloader_normal)
        # ----------------------------------------------------
        # 5. Create fresh ConvNeXt-Tiny
        # ----------------------------------------------------
        model = create_pretrained_model(model_name=MODEL_NAME,output_shape=len(class_names),freeze_backbone=False).to(device)
        # ----------------------------------------------------
        # 6. Optimizer
        # ----------------------------------------------------
        optimizer = torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
        #----------------------------------------------------
        # 7. Loss function
        # ----------------------------------------------------
        if use_focal:
            # Strategies 3 and 4
            loss_fn = FocalLoss(gamma=2.0,weight=class_weights)
            loss_name = "weighted_focal_gamma_2"
            
        elif strategy_name == "ce_normal":
            # Strategy 1: same approach as 5.9
            loss_fn = nn.CrossEntropyLoss(weight=class_weights)
            loss_name = "weighted_cross_entropy"
    
        else:
            # Strategies  2
            loss_fn = nn.CrossEntropyLoss()
            loss_name = "ordinary_cross_entropy"
        # ----------------------------------------------------
        # 8. Fresh metrics for this trial
        # ----------------------------------------------------
        train_metrics_trial, val_metrics_trial = (create_metrics())
        # ----------------------------------------------------
        # 9. W&B run
        # ----------------------------------------------------
        run_name = (
            f"{MODEL_NAME}_"
            f"{strategy_name}_"
            f"trial_{trial.number}"
        )
        uses_class_weights = use_focal or strategy_name == "ce_normal"
        wandb.init(
            project="rock-classification",
            name=run_name,
            group=f"optuna_{strategy_name}",
            config={
                "model": MODEL_NAME,
                "strategy": strategy_name,
                "optuna_trial": trial.number,
                "epochs": OPTUNA_EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": lr,
                "weight_decay": weight_decay,
                "loss": loss_name,
                "gamma": (2.0 if use_focal else None),
                "class_weighted": uses_class_weights,
                "class_weight_formula": ("1/sqrt(class_count)"if uses_class_weights else "none"),
                "sampler": ("WeightedRandomSampler" if use_sampler else "normal"),
                "sampler_weight_formula": ("1/class_count" if use_sampler else "none"),
                "sampler_replacement": (True if use_sampler else False),
                "optimizer": "Adam",
                "freeze_backbone": False
            },
            reinit=True
        )
        try:
            # ------------------------------------------------
            # 10. Train
            # ------------------------------------------------
            results = train_test(
                model=model,

                train_dataloader=train_dataloader,
                test_dataloader=val_dataloader,

                optimizer=optimizer,

                train_metrics=train_metrics_trial,
                test_metrics=val_metrics_trial,

                device=device,
                class_names=class_names,

                loss_fn=loss_fn,

                epochs=OPTUNA_EPOCHS
            )


            # ------------------------------------------------
            # 11. BEST validation macro-F1
            #
            #     IMPORTANT:
            #     Optuna optimizes the best epoch,
            #     not simply the final epoch.
            # ------------------------------------------------

            val_f1 = np.asarray(results["test_f1"])
            best_epoch_idx = int(np.argmax(val_f1))
            best_epoch = best_epoch_idx + 1
            best_val_f1 = float(results["test_f1"][best_epoch_idx])
            best_val_acc = float(results["test_acc"][best_epoch_idx])
            best_val_precision = float(results["test_precision"][best_epoch_idx])
            best_val_recall = float(results["test_recall"][best_epoch_idx])
            train_f1_at_best_epoch = float(results["train_f1"][best_epoch_idx])
            # ------------------------------------------------
            # 12. Save useful information to Optuna
            # -----------------------------------------------
            trial.set_user_attr("strategy",strategy_name)
            trial.set_user_attr("best_epoch",best_epoch)
            trial.set_user_attr("best_val_f1",best_val_f1)
            trial.set_user_attr("best_val_accuracy",best_val_acc)
            trial.set_user_attr("best_val_precision",best_val_precision)
            trial.set_user_attr("best_val_recall",best_val_recall)
            trial.set_user_attr("train_f1_at_best_epoch", train_f1_at_best_epoch)
            # ------------------------------------------------
            # 13. Add best metrics to W&B summary
            # ------------------------------------------------

            wandb.run.summary["best_val_f1" ] = best_val_f1
            wandb.run.summary["best_epoch"] = best_epoch
            wandb.run.summary["best_val_accuracy"] = best_val_acc
            wandb.run.summary["best_val_precision"] = best_val_precision
            wandb.run.summary["best_val_recall"] = best_val_recall
            wandb.run.summary["train_f1_at_best_epoch"] = train_f1_at_best_epoch

            print("\n" + "-" * 50)
            print(f"Best epoch       : {best_epoch}")
            print(f"Best val F1      : {best_val_f1:.4f}")
            print(f"Val accuracy     : {best_val_acc:.4f}")
            print(f"Val precision    : {best_val_precision:.4f}")
            print(f"Val recall       : {best_val_recall:.4f}")
            print(
                f"Train F1 @ best  : "
                f"{train_f1_at_best_epoch:.4f}"
            )
            print("-" * 50)
            # ------------------------------------------------
            # 14. Return metric Optuna should MAXIMIZE
            # ------------------------------------------------
            # Log validation plots to the current trial's W&B run.
            for log_function in (log_confusion_matrix,log_f1_by_class,log_recall_by_class,log_confusion_matrix_with_distribution,):
                log_function(model=model,dataloader=val_dataloader,class_names=class_names,device=device,)
            
            log_training_images_vs_f1(model=model,train_dataloader=train_dataloader_normal,val_dataloader=val_dataloader,class_names=class_names,device=device,)
            
            wandb.run.summary["diagnostic_epoch"] = len(results["test_f1"])
            return best_val_f1
        finally:
            # ------------------------------------------------
            # 15. Finish W&B and clear GPU memory
            # ------------------------------------------------
            if wandb.run is not None:
                wandb.finish()

            del model
            del optimizer
            del loss_fn
            del train_metrics_trial
            del val_metrics_trial

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


    return objective


# ============================================================
# 12. Run the FOUR Optuna studies
# ============================================================
studies = {}

for strategy_name in STRATEGIES.keys():

    print("\n")
    print("#" * 80)
    print(
        f"STARTING OPTUNA STUDY: "
        f"{strategy_name.upper()}"
    )
    print("#" * 80)


    study_name = (
        f"{MODEL_NAME}_"
        f"{strategy_name}_"
        f"{STUDY_TAG}"
    )


    study = optuna.create_study(
        study_name=study_name,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        storage=STORAGE,
        load_if_exists=True
    )
    study.optimize(
        make_objective(strategy_name),
        n_trials=N_TRIALS_PER_STRATEGY,
        gc_after_trial=True
    )
    studies[strategy_name] = study
# ============================================================
# 13. Compare the best result from each strategy
# ============================================================
study_summary = []
for strategy_name, study in studies.items():
    best_trial = study.best_trial
    study_summary.append({
        "strategy": strategy_name,
        "best_val_f1":best_trial.value,
        "best_epoch":best_trial.user_attrs.get("best_epoch"),
        "val_accuracy":best_trial.user_attrs.get("best_val_accuracy"),
        "val_precision":best_trial.user_attrs.get("best_val_precision"),
        "val_recall":best_trial.user_attrs.get("best_val_recall"),
        "train_f1_at_best_epoch":best_trial.user_attrs.get("train_f1_at_best_epoch"),
        "learning_rate":best_trial.params.get("lr"),
        "weight_decay":best_trial.params.get("weight_decay"),
        "trial_number":best_trial.number
    })

study_summary_df = (
    pd.DataFrame(study_summary)
    .sort_values(
        "best_val_f1",
        ascending=False
    )
    .reset_index(drop=True)
)


study_summary_df

### 6.2.2.0 Fine tune runs from 6.2.2
fine tune trial 7 from strategy 2

#### 1. optuna objective function

In [ ]:
import copy
import gc
import random
from functools import partial
from time import perf_counter

import numpy as np
import optuna
import torch
import torch.nn as nn
import wandb
from tqdm.auto import tqdm

from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)

from modular.data_setup import (
    RockClassificationDataset,
    remove_bottom_annotation,
)
from modular.train_test import train_step, test_step
from modular.utility import (
    log_confusion_matrix,
    log_f1_by_class,
    log_recall_by_class,
    log_confusion_matrix_with_distribution,
)
# ============================================================
# Fixed configuration
# ============================================================

MODEL_NAME = "convnext_tiny"
OPTUNA_EPOCHS = 20
NUM_WORKERS = 4
SEED = 42

REFERENCE_TRIAL = 7       # Change to 2 to use trial 2's values.
TUNE_WEIGHT_DECAY = True  # False fixes weight decay too.

REFERENCE_PARAMS = {
    2: {
        "lr": 1.841072920573867e-05,
        "weight_decay": 5.407712220288122e-06,
    },
    7: {
        "lr": 2.0366442026830887e-05,
        "weight_decay": 7.274653135669419e-06,
    },
}

REFERENCE = REFERENCE_PARAMS[REFERENCE_TRIAL]
INITIAL_LR = REFERENCE["lr"]

BATCH_SIZES = [16, 32]
AUGMENTATIONS = ["current", "strong"]

# Use the full merged training and validation splits.
source_train = RockClassificationDataset(
    root_dir=train_dir,
    image_transform=None,
)
source_val = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=None,
)

STUDY_CLASS_NAMES = source_train.classes
NUM_CLASSES = len(STUDY_CLASS_NAMES)

assert NUM_CLASSES == 18, "Expected the merged 18-class dataset."
assert source_train.class_to_idx == source_val.class_to_idx
assert "class1" not in STUDY_CLASS_NAMES

# Calculate sampling weights from THIS full training dataset.
study_labels = np.asarray([label for _, label in source_train.samples],dtype=np.int64,)
study_counts = np.bincount(study_labels,minlength=NUM_CLASSES,)

if np.any(study_counts == 0):
    raise ValueError("Some training classes have no samples.")

study_sample_weights = torch.tensor(1.0 / study_counts[study_labels],dtype=torch.double,)

print("Training images:", len(source_train))
print("Validation images:", len(source_val))
print("Classes:", NUM_CLASSES)

# ============================================================
# Helpers
# ============================================================

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_trial_loaders(batch_size, augmentation):
    weights = get_pretrained_weights(MODEL_NAME)
    preprocessing = weights.transforms()

    remove_annotation = transforms.Lambda(
        partial(remove_bottom_annotation, pixels=90)
    )

    if augmentation == "current":
        # Matches your existing pretrained training pipeline.
        spatial_transforms = [
            transforms.Resize(
                preprocessing.resize_size,
                interpolation=preprocessing.interpolation,
                antialias=preprocessing.antialias,
            ),
            transforms.RandomCrop(preprocessing.crop_size),
        ]
        color_jitter = transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.03,
        )

    else:
        # Stronger cropping and color augmentation.
        spatial_transforms = [
            transforms.RandomResizedCrop(
                preprocessing.crop_size,
                scale=(0.70, 1.0),
                ratio=(0.90, 1.10),
                interpolation=preprocessing.interpolation,
                antialias=preprocessing.antialias,
            ),
        ]
        color_jitter = transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.15,
            hue=0.03,
        )

    train_transform = transforms.Compose([
        remove_annotation,
        *spatial_transforms,
        transforms.RandomHorizontalFlip(p=0.5),
        color_jitter,
        transforms.ToTensor(),
        transforms.Normalize(
            mean=preprocessing.mean,
            std=preprocessing.std,
        ),
    ])

    # Identical validation preprocessing for every trial.
    val_transform = transforms.Compose([
        remove_annotation,
        preprocessing,
    ])

    trial_train = copy.copy(source_train)
    trial_train.image_transform = train_transform

    trial_val = copy.copy(source_val)
    trial_val.image_transform = val_transform

    # Strategy 2: inverse-frequency sampling with replacement.
    sampler = WeightedRandomSampler(
        weights=study_sample_weights,
        num_samples=len(trial_train),
        replacement=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(
        trial_train,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED),
    )

    val_loader = DataLoader(
        trial_val,
        batch_size=32,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED + 1),
    )

    return train_loader, val_loader


def make_trial_metrics():
    metrics = MetricCollection({
        "accuracy": MulticlassAccuracy(
            num_classes=NUM_CLASSES, average="micro"
        ),
        "f1": MulticlassF1Score(
            num_classes=NUM_CLASSES, average="macro"
        ),
        "precision": MulticlassPrecision(
            num_classes=NUM_CLASSES, average="macro"
        ),
        "recall": MulticlassRecall(
            num_classes=NUM_CLASSES, average="macro"
        ),
    })

    return (
        metrics.clone(prefix="train_").to(device),
        metrics.clone(prefix="test_").to(device),
    )


# ============================================================
# Optuna objective
# ============================================================

def objective_strategy2_adamw(trial):
    batch_size = trial.suggest_categorical(
        "batch_size", BATCH_SIZES
    )
    augmentation = trial.suggest_categorical(
        "augmentation", AUGMENTATIONS
    )

    if TUNE_WEIGHT_DECAY:
        weight_decay = trial.suggest_float(
            "weight_decay", 1e-6, 1e-2, log=True
        )
    else:
        weight_decay = REFERENCE["weight_decay"]

    # Fixed INITIAL learning rate; cosine changes it each epoch.
    lr = INITIAL_LR

    model = optimizer = scheduler = loss_fn = None
    train_loader = val_loader = None
    trial_train_metrics = trial_val_metrics = None
    run = None

    try:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        train_loader, val_loader = make_trial_loaders(
            batch_size=batch_size,
            augmentation=augmentation,
        )

        # Fresh pretrained model and classifier for every trial.
        model = create_pretrained_model(
            model_name=MODEL_NAME,
            output_shape=NUM_CLASSES,
            freeze_backbone=False,
        ).to(device)

        assert all(p.requires_grad for p in model.parameters())

        # Fresh optimizer and scheduler for every trial.
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=OPTUNA_EPOCHS,
            eta_min=lr * 0.01,
        )

        # Strategy 2: no class weighting in the loss.
        loss_fn = nn.CrossEntropyLoss()

        trial_train_metrics, trial_val_metrics = make_trial_metrics()

        

        run = wandb.init(
            project="rock-classification",
            group=trial.study.study_name,
            name=f"strategy2_adamw_trial_{trial.number}",
            tags=["merged18", "strategy2", "adamw", "cosine"],
            config={
                "model": MODEL_NAME,
                "strategy": "ce_weighted_sampler",
                "optuna_trial": trial.number,
                "reference_trial": REFERENCE_TRIAL,
                "optimizer": "AdamW",
                "initial_learning_rate": lr,
                "weight_decay": weight_decay,
                "tune_weight_decay": TUNE_WEIGHT_DECAY,
                "scheduler": "CosineAnnealingLR",
                "minimum_learning_rate": lr * 0.01,
                "epochs": OPTUNA_EPOCHS,
                "batch_size": batch_size,
                "augmentation": augmentation,
                "loss": "ordinary_cross_entropy",
                "class_weighted": False,
                "sampler_weight_formula": "1/class_count",
                "sampler_replacement": True,
                "freeze_backbone": False,
                "training_images": len(source_train),
                "num_classes": NUM_CLASSES,
                "class_names": STUDY_CLASS_NAMES,
                "train_dir": str(train_dir),
                "val_dir": str(val_dir),
                "seed": SEED,
            },
        )

        best_f1 = -float("inf")
        best_summary = {}
        started = perf_counter()
        
        run.define_metric("epoch")
        run.define_metric("train/*", step_metric="epoch")
        run.define_metric("val/*", step_metric="epoch")
        run.define_metric("learning_rate", step_metric="epoch")


        for epoch in tqdm(range(1, OPTUNA_EPOCHS + 1),desc=f"Trial {trial.number}",unit="epoch",):
            epoch_started = perf_counter()
            lr_used = optimizer.param_groups[0]["lr"]

            train_loss, train_stats = train_step(
                model=model,
                train_dataloader=train_loader,
                loss_fn=loss_fn,
                optimizer=optimizer,
                metrics=trial_train_metrics,
                device=device,
            )

            val_loss, val_stats = test_step(
                model=model,
                test_dataloader=val_loader,
                loss_fn=loss_fn,
                metrics=trial_val_metrics,
                device=device,
            )

            val_f1 = val_stats["test_f1"].item()

            if not np.isfinite(val_f1):
                raise RuntimeError("Validation F1 is not finite.")

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_summary = {
                    "best_epoch": epoch,
                    "best_val_f1": best_f1,
                    "best_val_accuracy":
                        val_stats["test_accuracy"].item(),
                    "best_val_precision":
                        val_stats["test_precision"].item(),
                    "best_val_recall":
                        val_stats["test_recall"].item(),
                    "train_f1_at_best_epoch":
                        train_stats["train_f1"].item(),
                }

            log_values = {
                "epoch": epoch,
                "epoch_time": perf_counter() - epoch_started,
                "learning_rate": lr_used,
                "train/loss": train_loss,
                "val/loss": val_loss,
            }

            for metric in ["accuracy", "f1", "precision", "recall"]:
                log_values[f"train/{metric}"] = (
                    train_stats[f"train_{metric}"].item()
                )
                log_values[f"val/{metric}"] = (
                    val_stats[f"test_{metric}"].item()
                )

            run.log(log_values)

            print(
                f"Trial {trial.number} | Epoch {epoch:02d} | "
                f"LR {lr_used:.3e} | "
                f"Train F1 {train_stats['train_f1'].item():.4f} | "
                f"Val F1 {val_f1:.4f} | Best {best_f1:.4f}"
            )

            # Step AFTER the epoch's optimizer updates.
            scheduler.step()

        for key, value in best_summary.items():
            trial.set_user_attr(key, value)
            run.summary[key] = value

        trial.set_user_attr("initial_learning_rate", lr)
        trial.set_user_attr("weight_decay", weight_decay)
        trial.set_user_attr("training_time", perf_counter() - started)

        # These plots describe the final epoch, as in your current study.
        for log_function in (
            log_confusion_matrix,
            log_f1_by_class,
            log_recall_by_class,
            log_confusion_matrix_with_distribution,
        ):
            log_function(
                model=model,
                dataloader=val_loader,
                class_names=STUDY_CLASS_NAMES,
                device=device,
            )

        run.summary["diagnostic_epoch"] = OPTUNA_EPOCHS

        # No pruning: every successful trial completes all epochs.
        return best_f1

    finally:
        if run is not None:
            run.finish()

        model = optimizer = scheduler = loss_fn = None
        train_loader = val_loader = None
        trial_train_metrics = trial_val_metrics = None

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

#### 2. Create the Optuna study

In [ ]:
# A separate study for the changed optimizer and training recipe.
mode = "tune_wd" if TUNE_WEIGHT_DECAY else "fixed_wd"

study_name = (
    f"convnext_strategy2_adamw_cosine_merged18_"
    f"ref{REFERENCE_TRIAL}_{mode}_{OPTUNA_EPOCHS}epochs_v1"
)

if TUNE_WEIGHT_DECAY:
    sampler = optuna.samplers.TPESampler(seed=SEED)
    n_trials = 20
else:
    # Both LR and weight decay fixed:
    # test all 2 batch sizes × 2 augmentation recipes.
    sampler = optuna.samplers.GridSampler(
        search_space={
            "batch_size": BATCH_SIZES,
            "augmentation": AUGMENTATIONS,
        },
        seed=SEED,
    )
    n_trials = len(BATCH_SIZES) * len(AUGMENTATIONS)

study_adamw = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    sampler=sampler,
    storage="sqlite:///convnext_strategy2_adamw_optuna.db",
    load_if_exists=True,
)

if TUNE_WEIGHT_DECAY:
    # Include the reference trial's decay value explicitly,
    # first with current augmentation, then stronger augmentation.
    for augmentation in AUGMENTATIONS:
        study_adamw.enqueue_trial(
            {
                "weight_decay": REFERENCE["weight_decay"],
                "batch_size": 32,
                "augmentation": augmentation,
            },
            skip_if_exists=True,
        )

study_adamw.optimize(
    objective_strategy2_adamw,
    n_trials=n_trials,
    gc_after_trial=True,
)

best_trial = study_adamw.best_trial

print("Best trial:", best_trial.number)
print("Best validation macro-F1:", best_trial.value)
print("Best epoch:", best_trial.user_attrs["best_epoch"])
print("Initial LR:", best_trial.user_attrs["initial_learning_rate"])
print("Weight decay:", best_trial.user_attrs["weight_decay"])
print("Batch size:", best_trial.params["batch_size"])
print("Augmentation:", best_trial.params["augmentation"])

### 6.2.3 Get the best model out

In [ ]:


import optuna

storage = (
    "sqlite:////home/d/Documents/Other_projects/"
    "Rock_Classification/convnext_4strategy_optuna.db"
)

strategy_names = [
    "ce_normal",
    "ce_weighted_sampler",
    "weighted_focal_normal",
    "weighted_focal_weighted_sampler",
]

studies = {
    name: optuna.load_study(
        study_name=f"{MODEL_NAME}_{name}_{STUDY_TAG}",
        storage=storage,
    )
    for name in strategy_names
}

# Add the new AdamW + cosine study.
studies["ce_weighted_sampler_adamw_cosine"] = optuna.load_study(
    study_name=(
        "convnext_strategy2_adamw_cosine_merged18_"
        "ref7_tune_wd_20epochs_v1"
    ),
    storage=(
        "sqlite:////home/d/Documents/Other_projects/"
        "Rock_Classification/convnext_strategy2_adamw_optuna.db"
    ),
)

In [ ]:
# ============================================================
# Find the BEST trial across all loaded studies
# ============================================================

best_strategy_name = max(studies,key=lambda name: studies[name].best_value)

best_study = studies[best_strategy_name]
best_trial = best_study.best_trial

# Original studies tuned LR; the AdamW study fixed it.
if "lr" in best_trial.params:
    best_lr = best_trial.params["lr"]
else:
    best_lr = best_trial.user_attrs["initial_learning_rate"]

# Supports both tuned and fixed weight decay.
if "weight_decay" in best_trial.params:
    best_weight_decay = best_trial.params["weight_decay"]
else:
    best_weight_decay = best_trial.user_attrs["weight_decay"]

best_epoch = best_trial.user_attrs["best_epoch"]
best_val_f1 = best_trial.value

# Original studies used batch size 32 and current augmentation.
best_batch_size = best_trial.params.get("batch_size", 32)
best_augmentation = best_trial.params.get("augmentation", "current")

is_adamw = (
    best_strategy_name == "ce_weighted_sampler_adamw_cosine"
)
best_optimizer_name = "AdamW" if is_adamw else "Adam"
best_scheduler_name = "CosineAnnealingLR" if is_adamw else "None"

print("=" * 70)
print("BEST OVERALL OPTUNA RESULT")
print("=" * 70)

print(f"Strategy       : {best_strategy_name}")
print(f"Trial number   : {best_trial.number}")
print(f"Validation F1  : {best_val_f1:.6f}")
print(f"Best epoch     : {best_epoch}")
print(f"Initial LR     : {best_lr:.6g}")
print(f"Weight decay   : {best_weight_decay:.6g}")
print(f"Batch size     : {best_batch_size}")
print(f"Augmentation   : {best_augmentation}")
print(f"Optimizer      : {best_optimizer_name}")
print(f"Scheduler      : {best_scheduler_name}")

### 6.2.3 Recreate the winning training setup
change the best epoch to to 5

In [ ]:
sample_weights = torch.tensor(
    1.0 / class_counts[np.asarray(train_labels)],
    dtype=torch.double,
)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=torch.Generator().manual_seed(42),
)

In [ ]:
# ============================================================
# Experiment configuration: reproduce AdamW trial 21
# ============================================================

MODEL_NAME = "convnext_tiny"
SEED = 42
NUM_WORKERS = 4

best_lr = 2.0366442026830887e-05
best_weight_decay = 0.0014810456320082151

weights = get_pretrained_weights(MODEL_NAME)

# ---------------------------------------------------------
# 1. Prepare trial 21's data
# ---------------------------------------------------------
train_dataloader, val_dataloader = make_trial_loaders(batch_size=32,augmentation="current",)

# ---------------------------------------------------------
# 2. Model summary
# ---------------------------------------------------------
model_check = create_pretrained_model(
    model_name=MODEL_NAME,
    output_shape=len(STUDY_CLASS_NAMES),
    freeze_backbone=False,
).to(device)

model_summary(model=model_check, weights=weights)
del model_check

# Reset seeds AFTER the temporary model.
# Create the actual training model immediately after this.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
# ---------------------------------------------------------
# 3. Reproduce AdamW trial 21 through its best epoch
# ---------------------------------------------------------
import random
import numpy as np
import torch
import wandb

from time import perf_counter
from tqdm.auto import tqdm
from modular.train_test import train_step, test_step

REPRODUCE_EPOCHS = 11
SCHEDULER_EPOCHS = 20  # Preserve the original trial's schedule.

best_lr = 2.0366442026830887e-05
best_weight_decay = 0.0014810456320082151

# Reset seeds before creating the actual model.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

best_model = create_pretrained_model(model_name=MODEL_NAME,output_shape=len(STUDY_CLASS_NAMES),freeze_backbone=False,).to(device)

optimizer = torch.optim.AdamW(best_model.parameters(),lr=best_lr,weight_decay=best_weight_decay,)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=SCHEDULER_EPOCHS,eta_min=best_lr * 0.01,)

# Strategy 2: class balancing is handled by the sampler.
loss_fn = torch.nn.CrossEntropyLoss()

# Fresh metrics using the helper from 6.2.2.0.
reproduce_train_metrics, reproduce_val_metrics = make_trial_metrics()

results = {
    key: []
    for key in [
        "train_loss", "train_acc", "train_f1",
        "train_precision", "train_recall",
        "test_loss", "test_acc", "test_f1",
        "test_precision", "test_recall",
        "epoch_time", "learning_rate",
    ]
}

run = wandb.init(
    project="rock-classification",
    name="convnext_tiny_reproduce_adamw_trial21_epoch11",
    tags=["merged18", "strategy2", "adamw", "reproduction"],
    config={
        "model": MODEL_NAME,
        "source_study": (
            "convnext_strategy2_adamw_cosine_merged18_"
            "ref7_tune_wd_20epochs_v1"
        ),
        "source_optuna_trial": 21,
        "optimizer": "AdamW",
        "initial_learning_rate": best_lr,
        "weight_decay": best_weight_decay,
        "epochs": REPRODUCE_EPOCHS,
        "scheduler": "CosineAnnealingLR",
        "scheduler_T_max": SCHEDULER_EPOCHS,
        "minimum_learning_rate": best_lr * 0.01,
        "batch_size": 32,
        "augmentation": "current",
        "loss": "ordinary_cross_entropy",
        "class_weighted": False,
        "sampler": "WeightedRandomSampler",
        "sampler_weight_formula": "1/class_count",
        "sampler_replacement": True,
        "freeze_backbone": False,
        "num_classes": len(STUDY_CLASS_NAMES),
        "seed": 42,
    },
)

run.define_metric("epoch")
run.define_metric("train/*", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")
run.define_metric("learning_rate", step_metric="epoch")

started = perf_counter()

try:
    for epoch in tqdm(
        range(1, REPRODUCE_EPOCHS + 1),
        desc="Reproduce trial 21",
        unit="epoch",
    ):
        epoch_started = perf_counter()
        lr_used = optimizer.param_groups[0]["lr"]

        train_loss, train_stats = train_step(
            model=best_model,
            train_dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            metrics=reproduce_train_metrics,
            device=device,
        )

        val_loss, val_stats = test_step(
            model=best_model,
            test_dataloader=val_dataloader,
            loss_fn=loss_fn,
            metrics=reproduce_val_metrics,
            device=device,
        )

        epoch_time = perf_counter() - epoch_started

        results["train_loss"].append(train_loss)
        results["test_loss"].append(val_loss)
        results["epoch_time"].append(epoch_time)
        results["learning_rate"].append(lr_used)

        log_values = {
            "epoch": epoch,
            "epoch_time": epoch_time,
            "learning_rate": lr_used,
            "train/loss": train_loss,
            "val/loss": val_loss,
        }

        for metric in ["accuracy", "f1", "precision", "recall"]:
            result_key = "acc" if metric == "accuracy" else metric
            train_value = train_stats[f"train_{metric}"].item()
            val_value = val_stats[f"test_{metric}"].item()

            results[f"train_{result_key}"].append(train_value)
            results[f"test_{result_key}"].append(val_value)

            log_values[f"train/{metric}"] = train_value
            log_values[f"val/{metric}"] = val_value

        run.log(log_values)

        tqdm.write(
            f"Epoch {epoch:02d} | LR {lr_used:.3e} | "
            f"Train F1 {results['train_f1'][-1]:.4f} | "
            f"Val F1 {results['test_f1'][-1]:.4f}"
        )

        scheduler.step()

    training_time = perf_counter() - started

    run.summary["training_time"] = training_time
    run.summary["best_val_f1"] = max(results["test_f1"])
    run.summary["best_epoch"] = (
        int(np.argmax(results["test_f1"])) + 1
    )
    run.summary["model_epoch"] = REPRODUCE_EPOCHS

    # Evaluate the model at epoch 11.
    for log_function in (
        log_confusion_matrix,
        log_f1_by_class,
        log_recall_by_class,
        log_confusion_matrix_with_distribution,
    ):
        log_function(
            model=best_model,
            dataloader=val_dataloader,
            class_names=STUDY_CLASS_NAMES,
            device=device,
        )

finally:
    run.finish()

best_model = best_model.cpu()
del optimizer, scheduler, loss_fn

# ---------------------------------------------------------
# 4. Store experiment
# ---------------------------------------------------------
experiment_name = f"{MODEL_NAME}_adamw_trial21_epoch11"

pretrained_model_experiments[experiment_name] = {
    "model": best_model,
    "results": results,
    "training_time": training_time,
    "weights": weights,  # Original pretrained weights enum.

    "source_optuna_trial": 21,
    "optimizer": "AdamW",
    "initial_learning_rate": best_lr,
    "weight_decay": best_weight_decay,
    "epochs": REPRODUCE_EPOCHS,
    "scheduler": "CosineAnnealingLR",
    "scheduler_T_max": SCHEDULER_EPOCHS,
    "scheduler_eta_min": best_lr * 0.01,

    "batch_size": 32,
    "augmentation": "current",
    "loss": "ordinary_cross_entropy",
    "sampling": "inverse_frequency",
    "freeze_backbone": False,
    "class_names": list(STUDY_CLASS_NAMES),
    "class_to_idx": dict(source_train.class_to_idx),
    "seed": 42,
}

### 6.2.4 save the model

In [ ]:
# Save the model for inference
import importlib
import modular.utility as utils
utils.save_model(model=best_model,target_dir="models", model_name="INFERENCE_MERGED__18_class_convnext_tiny_reproduce_adamw_trial21_epoch11.pth")

In [ ]:
# save full model
checkpoint = {
    "model_name": "convnext_tiny",
    "model_state_dict": best_model.state_dict(),

    "num_classes": len(class_names),
    "class_names": class_names,
    "class_to_idx": train_dataset.class_to_idx,

    "strategy": "weighted_focal_normal",
    "epochs": 5,
    "learning_rate": best_lr,
    "weight_decay": best_weight_decay,

    "test_metrics": final_test_metrics,
    "wandb_run_url": final_test_run_url,
}

torch.save(
    checkpoint,
    "models/FULL_V2_convnext_tiny_weighted_focal_trial9_5epochs.pth"
)

In [ ]:
from pathlib import Path
import torch

checkpoint = {
    "model_name": MODEL_NAME,
    "model_state_dict": best_model.state_dict(),

    "num_classes": len(STUDY_CLASS_NAMES),
    "class_names": list(STUDY_CLASS_NAMES),
    "class_to_idx": dict(source_train.class_to_idx),

    "strategy": "ce_weighted_sampler_adamw_cosine",
    "source_optuna_trial": 21,
    "epochs": REPRODUCE_EPOCHS,  # 11

    "optimizer": "AdamW",
    "initial_learning_rate": best_lr,
    "weight_decay": best_weight_decay,

    "scheduler": "CosineAnnealingLR",
    "scheduler_T_max": SCHEDULER_EPOCHS,  # 20
    "scheduler_eta_min": best_lr * 0.01,

    "batch_size": 32,
    "augmentation": "current",
    "remove_bottom_pixels": 90,
    "pretrained_weights": str(weights),
    "loss": "ordinary_cross_entropy",
    "sampling": "inverse_frequency",
    "sampler_replacement": True,
    "seed": 42,

    "validation_metrics_at_saved_epoch": {
        "accuracy": results["test_acc"][-1],
        "f1_macro": results["test_f1"][-1],
        "precision_macro": results["test_precision"][-1],
        "recall_macro": results["test_recall"][-1],
    },

    "training_history": results,
    "training_time": training_time,
    "training_wandb_run_url": run.url,
}

save_path = Path(
    "models/FULL_MERGED_18_classes_convnext_tiny_adamw_trial21_epoch11.pth"
)
save_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(checkpoint, save_path)
print(f"Saved: {save_path}")

# 7 Testing

Evaluate the trained five-epoch ConvNeXt-Tiny from section 6.2.3 on the test split.
Log one set of overall test metrics, confusion matrices, and per-class precision,
recall, and F1 to a separate W&B run tagged `v2` and `test`. No training epochs run here.


In [ ]:
#optionally load the model
import torch
from torchvision.models import convnext_tiny

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    "/home/d/Documents/Other_projects/Rock_Classification/models/"
    "FULL_MERGED_18_classes_convnext_tiny_adamw_trial21_epoch11.pth",
    map_location="cpu",
    weights_only=True,
)

# Restore the saved class order
class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

# Recreate the architecture
best_model = convnext_tiny(
    weights=None,
    num_classes=checkpoint["num_classes"],
)

# Restore your trained weights
best_model.load_state_dict(checkpoint["model_state_dict"])

# Prepare for evaluation
best_model = best_model.to(device)
best_model.eval()

print(f"Model loaded on {device}")

In [ ]:
import importlib
import modular.utility
import modular.train_test

importlib.reload(modular.utility)
importlib.reload(modular.train_test)

from modular.train_test import evaluate_model
from modular.utility import (log_confusion_matrix,log_f1_by_class,log_recall_by_class,log_precision_by_class,log_confusion_matrix_with_distribution,)
import wandb

## 7.1 Inference on train data

In [ ]:
from functools import partial
from torchvision import transforms
from torchvision.models import ConvNeXt_Tiny_Weights
from modular.data_setup import remove_bottom_annotation

train_eval_transform = transforms.Compose([
    transforms.Lambda(
        partial(remove_bottom_annotation, pixels=90)
    ),
    ConvNeXt_Tiny_Weights.IMAGENET1K_V1.transforms(),
])

In [ ]:
##########################################
# make train dataset again for inference
##########################################
train_eval_dataset = RockClassificationDataset(
    root_dir=train_dir,
    image_transform=train_eval_transform
)

train_eval_dataloader = DataLoader(
    train_eval_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

##########################################
# set run name
##########################################
run_name = "Final_result_train_merge"

wandb.init(project="rock-classification",name=run_name,tags=["merge"])

##########################################
# make training metrics
##########################################
metrics = metrics.to(device)

train_metrics = evaluate_model(
    model=best_model,
    dataloader=train_eval_dataloader,
    metrics=metrics,
    class_names=class_names,
    device=device
)

wandb.log({
    "final_train_accuracy": train_metrics["accuracy"],
    "final_train_f1_macro": train_metrics["f1"],
    "final_train_precision_macro": train_metrics["precision"],
    "final_train_recall_macro": train_metrics["recall"],
})

##########################################
# Log plots to W&B
##########################################
log_confusion_matrix(model=best_model,dataloader=train_eval_dataloader,class_names=class_names,device=device)

log_f1_by_class(model=best_model,dataloader=train_eval_dataloader,class_names=class_names,device=device)

log_recall_by_class(model=best_model,dataloader=train_eval_dataloader,class_names=class_names,device=device)
log_precision_by_class(model=best_model,dataloader=train_eval_dataloader,class_names=class_names,device=device,)

log_confusion_matrix_with_distribution(model=best_model,dataloader=train_eval_dataloader,class_names=class_names,device=device)

wandb.finish()

## 7.2 Inference on Val

In [ ]:
##########################################
# make train dataset again for inference
##########################################
val_eval_dataset = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=train_eval_transform
)

val_eval_dataloader = DataLoader(
    val_eval_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

##########################################
# set run name
##########################################
run_name = "Final_result_Val_merge"

wandb.init(project="rock-classification",name=run_name,tags=["merge"])

##########################################
# make training metrics
##########################################
metrics = metrics.to(device)

val_metrics = evaluate_model(
    model=best_model,
    dataloader=val_eval_dataloader,
    metrics=metrics,
    class_names=class_names,
    device=device
)

wandb.log({
    "final_val_accuracy": val_metrics["accuracy"],
    "final_val_f1_macro": val_metrics["f1"],
    "final_val_precision_macro": val_metrics["precision"],
    "final_val_recall_macro": val_metrics["recall"],
})

##########################################
# Log plots to W&B
##########################################
log_confusion_matrix(model=best_model,dataloader=val_eval_dataloader,class_names=class_names,device=device)

log_f1_by_class(model=best_model,dataloader=val_eval_dataloader,class_names=class_names,device=device)

log_recall_by_class(model=best_model,dataloader=val_eval_dataloader,class_names=class_names,device=device)
log_precision_by_class(model=best_model,dataloader=val_eval_dataloader,class_names=class_names,device=device,)

log_confusion_matrix_with_distribution(model=best_model,dataloader=val_eval_dataloader,class_names=class_names,device=device)

wandb.finish()

## 7.3 Inference on test

In [ ]:
##########################################
# make test dataset again for inference
##########################################
test_eval_dataset = RockClassificationDataset(
    root_dir=test_dir,
    image_transform=train_eval_transform
)

test_eval_dataloader = DataLoader(
    test_eval_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

##########################################
# set run name
##########################################
run_name = "Final_result_Test_merge"

wandb.init(project="rock-classification",name=run_name,tags=["merge"])

##########################################
# make test metrics
##########################################
metrics = metrics.to(device)

test_metrics = evaluate_model(
    model=best_model,
    dataloader=test_eval_dataloader,
    metrics=metrics,
    class_names=class_names,
    device=device
)

wandb.log({
    "final_test_accuracy": test_metrics["accuracy"],
    "final_test_f1_macro": test_metrics["f1"],
    "final_test_precision_macro": test_metrics["precision"],
    "final_test_recall_macro": test_metrics["recall"],
})

##########################################
# Log plots to W&B
##########################################
log_confusion_matrix(model=best_model,dataloader=test_eval_dataloader,class_names=class_names,device=device)

log_f1_by_class(model=best_model,dataloader=test_eval_dataloader,class_names=class_names,device=device)

log_recall_by_class(model=best_model,dataloader=test_eval_dataloader,class_names=class_names,device=device)
log_precision_by_class(model=best_model,dataloader=test_eval_dataloader,class_names=class_names,device=device,)

log_confusion_matrix_with_distribution(model=best_model,dataloader=test_eval_dataloader,class_names=class_names,device=device)

wandb.finish()

# Other

In [ ]:
# ============================================================
# Recreate winning strategy
# ============================================================

best_strategy = STRATEGIES[best_strategy_name]

use_sampler = best_strategy["use_sampler"]
use_focal = best_strategy["use_focal"]


# ------------------------------------------------------------
# Training DataLoader
# ------------------------------------------------------------

if use_sampler:

    final_train_dataloader = (
        create_weighted_sampler_dataloader(
            seed=SEED
        )
    )

else:

    final_train_dataloader = train_dataloader_normal


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

if use_focal:

    final_loss_fn = FocalLoss(
        gamma=2.0,
        weight=class_weights
    )

else:

    final_loss_fn = nn.CrossEntropyLoss()


print("Sampler:",
      "WeightedRandomSampler"
      if use_sampler else "Normal")

print("Loss:",
      "Weighted Focal Loss"
      if use_focal else "CrossEntropyLoss")

## Create a brand-new ConvNeXt-Tiny

In [ ]:
# ============================================================
# Create fresh final model
# ============================================================

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


final_model = create_pretrained_model(
    model_name=MODEL_NAME,
    output_shape=len(class_names),
    freeze_backbone=False
).to(device)


final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=best_lr,
    weight_decay=best_weight_decay
)

In [ ]:
final_train_metrics, final_val_metrics = create_metrics()

## Retrain for the winning number of epochs

In [ ]:
epochs=best_epoch

In [ ]:
# ============================================================
# Start W&B run for final retraining
# ============================================================

wandb.init(
    project="rock-classification",

    name=f"FINAL_{best_strategy_name}",

    group="final_model",

    config={
        "model": MODEL_NAME,
        "strategy": best_strategy_name,

        "learning_rate": best_lr,
        "weight_decay": best_weight_decay,

        "epochs": best_epoch,

        "sampler": (
            "WeightedRandomSampler"
            if use_sampler
            else "normal"
        ),

        "loss": (
            "weighted_focal_gamma_2"
            if use_focal
            else "cross_entropy"
        ),

        "gamma": (
            2.0
            if use_focal
            else 0.0
        )
    }
)

In [ ]:
# ============================================================
# Retrain final model
# ============================================================

final_results = train_test(

    model=final_model,

    train_dataloader=final_train_dataloader,
    test_dataloader=val_dataloader,

    optimizer=final_optimizer,

    train_metrics=final_train_metrics,
    test_metrics=final_val_metrics,

    device=device,
    class_names=class_names,

    loss_fn=final_loss_fn,

    epochs=best_epoch
)

### 5.1.2 Plot the loss curve of Model 0

### 5.1.3 model diagnostic Plot the loss curve of Model 0

In [ ]:
plot_model_diagnostics(
    model=model_0_0_tinyVGG,
    train_dataloader=train_dataloader_5_simple,
    test_dataloader=val_dataloader_5_simple,
    class_names=class_names,
    model_name="TinyVGG",
    device=device
)